# 02 - Data Preparation and Integration  
  
*AFL Matchday Demand FOrecasting - Preparing Reliable Match and Calendar Tables*

## 1. Objective and Scope

### 1.1 Notebook Objective

This notebook prepares the validated raw datasets.

It transforms source files with different formats, field names, and naming conventions into consistent, database-ready tables for later PostgreSQL loading and feature engineering.

The preparation focuses on three data groups:

- historical AFL match and attendance data;
- 2026 Squiggle match and team snapshots;
- historical and official school-holiday records.

The original source files remain unchanged. All transformations are applied to in-memory copies, and the validated results are exported separately.

### 1.2 Selected v1 Data Sources

Only data sources that directly support match-level demand forecasting or the preparation of 2026 scoring fixtures are included in version 1.

| Data source | Raw format | Purpose in this notebook |
|---|---|---|
| AFLStats `games.csv` | CSV | Provides historical match-level results, attendance, teams, venues, and round information. |
| `squiggle_games_2026_20260817.json` | JSON | Provides the 2026 match and fixture records available at the snapshot date. |
| `squiggle_teams_20260817.json` | JSON | Provides Squiggle team identifiers and names for consistent team mapping. |
| Historical school-holiday dataset from Figshare | R data file | Provides daily city-level school-holiday records for 2004–2023. |
| Official state and territory school calendars | PDF documents | Provide reviewed school-term and holiday periods for 2024–2026. |

The downloaded `players.csv` and `stats.csv` files are retained in the raw-data directory but are not processed in this notebook. Version 1 uses match-level predictors and does not include player-level features.

### 1.3 Expected Outputs and Processing Boundaries

This notebook prepares five structured datasets for later database loading and feature engineering.

| Prepared output | Record grain | Intended use |
|---|---|---|
| Historical match table | One row per completed historical match | Supports later feature engineering and model development. |
| Prepared 2026 match snapshot | One row per Squiggle match or scheduled fixture | Separates completed matches from future scoring fixtures. |
| Team reference table | One row per standardised AFL team | Provides consistent team names and identifiers across sources. |
| Venue reference table | One row per standardised venue | Provides consistent venue names and associated locations. |
| Daily school-holiday table | One row per covered city and date | Supports later calendar-feature creation through match date and venue location. |

Validated outputs will be exported to `data/interim/postgres_ready/`.

The processing included in this notebook is limited to:

- selecting required source fields;
- standardising column names and data types;
- aligning team and venue names;
- applying previously reviewed source corrections;
- structuring and combining school-holiday records;
- validating key fields, identifiers, mappings, and date coverage;
- exporting the prepared tables.

The following tasks are outside the scope of this notebook:

- processing player-level data;
- creating predictive features or attendance classes;
- splitting data into training, validation, and test periods;
- training or evaluating machine-learning models;
- loading data into PostgreSQL;
- collecting live weather forecasts;
- deploying or automating the workflow in AWS.

## 2. Load the Selected Data Sources

### 2.1 Historical Match Data

The historical match dataset is loaded from the AFLStats `games.csv` file reviewed in Notebook 01.

At this stage, the notebook confirms that the source can be loaded successfully and that the fields required for later preparation are available. The detailed raw-data audit is not repeated here.

**Load and initial confirmation**

The source CSV is loaded using a project-relative path so that the notebook does not depend on a specific local drive or user directory. Only a small preview and structural summary are displayed at this stage.

In [1]:
from pathlib import Path

import pandas as pd


# Identify the project root so the notebook remains portable when it is
# executed from either the project directory or the notebooks directory.
notebook_working_directory = Path.cwd()
project_root = (
    notebook_working_directory.parent
    if notebook_working_directory.name.lower() == "notebooks"
    else notebook_working_directory
)

# Construct the source path relative to the project root rather than
# relying on a machine-specific absolute path.
historical_match_path = (
    project_root
    / "data"
    / "raw"
    / "extracted"
    / "aflstats_v6"
    / "games.csv"
)

# Stop execution with a clear message if the expected audited source
# is unavailable or has been moved.
if not historical_match_path.is_file():
    raise FileNotFoundError(
        f"Historical match source was not found: {historical_match_path}"
    )

# Load the audited source into memory without modifying the original CSV.
# Disabling chunk-based type inference provides consistent initial dtypes.
historical_matches_raw = pd.read_csv(
    historical_match_path,
    low_memory=False,
)

# Display a concise structural summary for reproducibility and review.
print(f"Source: {historical_match_path.relative_to(project_root)}")
print(f"Rows: {len(historical_matches_raw):,}")
print(f"Columns: {historical_matches_raw.shape[1]}")
print("\nColumn names:")
print(historical_matches_raw.columns.tolist())

display(historical_matches_raw.head(3))

Source: data\raw\extracted\aflstats_v6\games.csv
Rows: 2,879
Columns: 22

Column names:
['GameId', 'Year', 'Round', 'Date', 'MaxTemp', 'MinTemp', 'Rainfall', 'Venue', 'StartTime', 'Attendance', 'HomeTeam', 'HomeTeamScoreQT', 'HomeTeamScoreHT', 'HomeTeamScore3QT', 'HomeTeamScoreFT', 'HomeTeamScore', 'AwayTeam', 'AwayTeamScoreQT', 'AwayTeamScoreHT', 'AwayTeamScore3QT', 'AwayTeamScoreFT', 'AwayTeamScore']


,GameId,Year,Round,Date,MaxTemp,MinTemp,Rainfall,Venue,StartTime,Attendance,...,HomeTeamScoreHT,HomeTeamScore3QT,HomeTeamScoreFT,HomeTeamScore,AwayTeam,AwayTeamScoreQT,AwayTeamScoreHT,AwayTeamScore3QT,AwayTeamScoreFT,AwayTeamScore
0,2012R0101,2012,Round 1,2012-03-24,24.0,12.2,0.0,Stadium Australia,7:20 PM,"38,203",...,3.3,3.4,5.70,37,Sydney,4.1,8.4,13.80,14.16,100
1,2012R0102,2012,Round 1,2012-03-29,25.7,9.7,0.0,MCG,7:45 PM,"78,285",...,5.6,10.7,12.90,81,Carlton,3.2,8.7,11.13,18.17,125
2,2012R0103,2012,Round 1,2012-03-30,27.4,9.7,0.6,MCG,7:50 PM,"78,466",...,10.6,14.1,20.17,137,Collingwood,2.7,7.9,12.16,16.19,115


**Required-field confirmation**

The following check confirms that the fields required for match-level preparation are present in the source schema. Optional source fields are retained in the loaded DataFrame but are not treated as required version 1 inputs.

In [2]:
# Define the source fields required for later match-level preparation.
# Quarter-by-quarter scores and weather observations remain available
# in the raw DataFrame but are not required by this loading check.
required_historical_columns = [
    "GameId",
    "Year",
    "Round",
    "Date",
    "Venue",
    "StartTime",
    "Attendance",
    "HomeTeam",
    "AwayTeam",
    "HomeTeamScore",
    "AwayTeamScore",
]

# Build a readable schema check before applying any transformations.
required_column_check = pd.DataFrame(
    {
        "column_name": required_historical_columns,
        "present_in_source": [
            column in historical_matches_raw.columns
            for column in required_historical_columns
        ],
        "source_dtype": [
            (
                str(historical_matches_raw[column].dtype)
                if column in historical_matches_raw.columns
                else None
            )
            for column in required_historical_columns
        ],
    }
)

display(required_column_check)

# Stop the workflow if a required field is missing, as subsequent
# preparation steps would otherwise produce misleading failures.
missing_required_columns = required_column_check.loc[
    ~required_column_check["present_in_source"],
    "column_name",
].tolist()

if missing_required_columns:
    raise ValueError(
        "Missing required historical match columns: "
        f"{missing_required_columns}"
    )

print("Required historical column check: PASSED")

,column_name,present_in_source,source_dtype
0,GameId,True,str
1,Year,True,int64
2,Round,True,str
3,Date,True,str
4,Venue,True,str
5,StartTime,True,str
6,Attendance,True,str
7,HomeTeam,True,str
8,AwayTeam,True,str
9,HomeTeamScore,True,int64


Required historical column check: PASSED


**Result**

The historical match source loaded successfully with 2,879 rows and 22 source columns. All 11 fields required for later match-level preparation are present.

`Date`, `StartTime`, and `Attendance` are currently stored as object-type values. Their conversion is intentionally deferred to Section 3.1, where column names and data types will be standardised together.

No source records or values were modified in this section.

### 2.2 Squiggle Games and Teams Snapshots

The Squiggle API snapshots capture the 2026 game records and team reference information available on 17 August 2026.

The games snapshot may contain both completed matches and scheduled fixtures. The teams snapshot provides the source identifiers and names needed to interpret those game records consistently.

This section loads the two saved JSON responses and confirms their basic structure. Match-status classification and name standardisation are deferred to Section 4.

**Load and verify the saved API responses**

The saved responses are loaded as JSON objects before their record collections are converted into DataFrames. Checking the expected top-level keys first makes any unexpected change in the response structure easier to identify.

In [3]:
import json


# Define both snapshot paths relative to the previously identified
# project root so that the notebook remains portable.
squiggle_games_path = (
    project_root
    / "data"
    / "raw"
    / "api_snapshots"
    / "squiggle_games_2026_20260817.json"
)

squiggle_teams_path = (
    project_root
    / "data"
    / "raw"
    / "api_snapshots"
    / "squiggle_teams_20260817.json"
)

# Confirm that both saved API responses are available before loading
# either file, preventing a partially loaded snapshot pair.
snapshot_paths = {
    "games": squiggle_games_path,
    "teams": squiggle_teams_path,
}

missing_snapshot_files = [
    name
    for name, path in snapshot_paths.items()
    if not path.is_file()
]

if missing_snapshot_files:
    raise FileNotFoundError(
        "Missing Squiggle snapshot files: "
        f"{missing_snapshot_files}"
    )

# Load the complete JSON payloads so their original response structure
# remains available for inspection.
with squiggle_games_path.open(
    mode="r",
    encoding="utf-8",
) as file:
    squiggle_games_payload = json.load(file)

with squiggle_teams_path.open(
    mode="r",
    encoding="utf-8",
) as file:
    squiggle_teams_payload = json.load(file)

# Validate the expected Squiggle response collections before
# converting their records into tabular form.
if not isinstance(squiggle_games_payload.get("games"), list):
    raise ValueError(
        "The games snapshot does not contain an expected 'games' list."
    )

if not isinstance(squiggle_teams_payload.get("teams"), list):
    raise ValueError(
        "The teams snapshot does not contain an expected 'teams' list."
    )

# Convert the record collections into DataFrames without changing
# field names or source values.
squiggle_games_raw = pd.json_normalize(
    squiggle_games_payload["games"]
)

squiggle_teams_raw = pd.json_normalize(
    squiggle_teams_payload["teams"]
)

# Report the loaded structures without performing preparation yet.
print(
    "Games source:",
    squiggle_games_path.relative_to(project_root),
)
print(
    f"Games records: {len(squiggle_games_raw):,}; "
    f"columns: {squiggle_games_raw.shape[1]}"
)
print("Games columns:")
print(squiggle_games_raw.columns.tolist())

print(
    "\nTeams source:",
    squiggle_teams_path.relative_to(project_root),
)
print(
    f"Team records: {len(squiggle_teams_raw):,}; "
    f"columns: {squiggle_teams_raw.shape[1]}"
)
print("Teams columns:")
print(squiggle_teams_raw.columns.tolist())

Games source: data\raw\api_snapshots\squiggle_games_2026_20260817.json
Games records: 218; columns: 26
Games columns:
['hscore', 'ateamid', 'hgoals', 'is_final', 'complete', 'date', 'winner', 'agoals', 'unixtime', 'ascore', 'hteamid', 'tz', 'timestr', 'updated', 'hbehinds', 'ateam', 'localtime', 'winnerteamid', 'is_grand_final', 'round', 'year', 'roundname', 'hteam', 'id', 'abehinds', 'venue']

Teams source: data\raw\api_snapshots\squiggle_teams_20260817.json
Team records: 18; columns: 6
Teams columns:
['retirement', 'name', 'id', 'abbrev', 'debut', 'logo']


**Required-field confirmation**

Squiggle defines `complete` as the estimated percentage of a match completed. In contrast, `is_final` identifies whether the match belongs to the finals series and, where applicable, its finals category.

Completion status will therefore be derived from `complete`, not from `is_final`, during the preparation stage.

The following checks confirm that the match, scheduling, team, score, and status fields required for later processing are present in both snapshots.

Source: [Squiggle API documentation](https://api.squiggle.com.au/)

In [4]:
# Define the Squiggle game fields required for fixture preparation,
# team mapping, time standardisation, and completion classification.
required_squiggle_game_columns = [
    "id",
    "year",
    "round",
    "date",
    "localtime",
    "tz",
    "hteamid",
    "hteam",
    "ateamid",
    "ateam",
    "venue",
    "complete",
    "is_final",
    "hscore",
    "ascore",
]

# Define the team-reference fields required to interpret the team
# identifiers and names contained in the games snapshot.
required_squiggle_team_columns = [
    "id",
    "name",
    "abbrev",
    "debut",
    "retirement",
]

# Identify missing fields before displaying or using either schema.
missing_game_columns = [
    column
    for column in required_squiggle_game_columns
    if column not in squiggle_games_raw.columns
]

missing_team_columns = [
    column
    for column in required_squiggle_team_columns
    if column not in squiggle_teams_raw.columns
]

# Stop the workflow with a source-specific message if either saved
# response does not contain the required fields.
if missing_game_columns:
    raise ValueError(
        "Missing required Squiggle game columns: "
        f"{missing_game_columns}"
    )

if missing_team_columns:
    raise ValueError(
        "Missing required Squiggle team columns: "
        f"{missing_team_columns}"
    )

# Create readable schema summaries for the required fields only.
squiggle_game_column_check = pd.DataFrame(
    {
        "column_name": required_squiggle_game_columns,
        "present_in_source": True,
        "source_dtype": [
            str(squiggle_games_raw[column].dtype)
            for column in required_squiggle_game_columns
        ],
    }
)

squiggle_team_column_check = pd.DataFrame(
    {
        "column_name": required_squiggle_team_columns,
        "present_in_source": True,
        "source_dtype": [
            str(squiggle_teams_raw[column].dtype)
            for column in required_squiggle_team_columns
        ],
    }
)

print("Required Squiggle game fields:")
display(squiggle_game_column_check)

print("Required Squiggle team fields:")
display(squiggle_team_column_check)

print("Required Squiggle column checks: PASSED")

Required Squiggle game fields:


,column_name,present_in_source,source_dtype
0,id,True,int64
1,year,True,int64
2,round,True,int64
3,date,True,str
4,localtime,True,str
5,tz,True,str
6,hteamid,True,float64
7,hteam,True,str
8,ateamid,True,float64
9,ateam,True,str


Required Squiggle team fields:


,column_name,present_in_source,source_dtype
0,id,True,int64
1,name,True,str
2,abbrev,True,str
3,debut,True,int64
4,retirement,True,int64


Required Squiggle column checks: PASSED


**Compact source preview**

A limited set of fields is displayed to confirm the meaning and alignment of the two API responses without producing an unnecessarily wide notebook output. No rows are filtered or transformed.

In [5]:
# Select a compact set of game fields that represents identity,
# scheduling, participating teams, venue, and source status.
squiggle_game_preview_columns = [
    "id",
    "year",
    "round",
    "date",
    "hteam",
    "ateam",
    "venue",
    "complete",
    "is_final",
]

# Select the team fields needed to interpret the source identifiers
# used by the games snapshot.
squiggle_team_preview_columns = [
    "id",
    "name",
    "abbrev",
    "debut",
    "retirement",
]

print("Squiggle games snapshot preview:")
display(
    squiggle_games_raw.loc[
        :,
        squiggle_game_preview_columns,
    ].head(3)
)

print("Squiggle teams snapshot preview:")
display(
    squiggle_teams_raw.loc[
        :,
        squiggle_team_preview_columns,
    ].head(3)
)

Squiggle games snapshot preview:


,id,year,round,date,hteam,ateam,venue,complete,is_final
0,38494,2026,0,2026-03-05 19:30:00,Sydney,Carlton,S.C.G.,100,0
1,38495,2026,0,2026-03-06 20:05:00,Gold Coast,Geelong,Carrara,100,0
2,38496,2026,0,2026-03-07 16:15:00,Greater Western Sydney,Hawthorn,Sydney Showground,100,0


Squiggle teams snapshot preview:


,id,name,abbrev,debut,retirement
0,1,Adelaide,ADE,1991,9999
1,2,Brisbane Lions,BRI,1987,9999
2,3,Carlton,CAR,1897,9999


**Result**

The saved Squiggle games response loaded successfully with 218 records and 26 source fields. The teams response loaded successfully with 18 records and 6 source fields. All fields required for later preparation are present.

The games snapshot provides match identifiers, scheduling details, participating teams, venues, scores, completion progress, and finals categories. The teams snapshot provides the source identifiers and names needed to interpret the team fields consistently.

No records were filtered, classified, or standardised in this section. Completed matches, confirmed future fixtures, and fixtures with teams not yet determined will be distinguished in Section 4.

### 2.3 Historical and Reviewed School-Calendar Sources

The school-holiday inputs are stored in two different forms.

The historical Figshare source provides structured daily holiday records for 2004–2023. Official school-calendar documents from the relevant Australian state and territory authorities provide the source evidence needed to extend coverage through 2024–2026.

This section confirms that both source groups are available and loads only the existing structured historical data. The official PDF documents remain unchanged and are not treated as database-ready tables. Their reviewed date periods will be structured later in Section 5.

**Load the historical data and inventory the official documents**

An RDA file can contain one or more named R objects rather than a single visible table. It is therefore loaded as an object collection before the relevant DataFrame is selected.

The official calendar PDFs are inventoried but not parsed in this section. They remain source documents for the reviewed date-period preparation in Section 5.

In [6]:
import pyreadr


# Define the historical RDA source and the directory containing the
# official calendar documents using project-relative paths.
historical_holiday_path = (
    project_root
    / "data"
    / "raw"
    / "extracted"
    / "school_holidays_figshare_v1"
    / "school.holidays.rda"
)

official_calendar_directory = (
    project_root
    / "data"
    / "raw"
    / "source_documents"
    / "school_calendars_official_2024_2026"
)

# Define the official source documents expected from the completed
# collection step, including the two separate NSW documents.
expected_official_calendar_files = {
    "act_school_terms_2024_2026.pdf",
    "nsw_school_terms_2024_2025.pdf",
    "nsw_school_terms_2026_eastern.pdf",
    "nt_school_terms_2024_2026.pdf",
    "qld_school_terms_2024_2026.pdf",
    "sa_school_terms_2024_2026.pdf",
    "tas_school_terms_2024_2026.pdf",
    "vic_school_terms_2024_2026.pdf",
    "wa_school_terms_2024_2026.pdf",
}

# Confirm that the structured historical source and the official
# document directory are both available.
if not historical_holiday_path.is_file():
    raise FileNotFoundError(
        f"Historical holiday source was not found: "
        f"{historical_holiday_path}"
    )

if not official_calendar_directory.is_dir():
    raise FileNotFoundError(
        f"Official calendar directory was not found: "
        f"{official_calendar_directory}"
    )

# Inventory the available PDF files without extracting or changing
# their contents at this stage.
official_calendar_paths = sorted(
    official_calendar_directory.glob("*.pdf")
)

available_official_calendar_files = {
    path.name
    for path in official_calendar_paths
}

missing_official_calendar_files = sorted(
    expected_official_calendar_files
    - available_official_calendar_files
)

if missing_official_calendar_files:
    raise FileNotFoundError(
        "Missing official school-calendar documents: "
        f"{missing_official_calendar_files}"
    )

# Load the RDA container while preserving its original object names.
historical_holiday_objects = pyreadr.read_r(
    str(historical_holiday_path)
)

if not historical_holiday_objects:
    raise ValueError(
        "The historical school-holiday RDA file contains no readable objects."
    )

# Report the R objects and official documents available for later use.
print(
    "Historical source:",
    historical_holiday_path.relative_to(project_root),
)
print("R objects found:")

for object_name, object_data in historical_holiday_objects.items():
    print(
        f"- {object_name!r}: "
        f"{object_data.shape[0]:,} rows × "
        f"{object_data.shape[1]} columns"
    )

print(
    "\nOfficial calendar directory:",
    official_calendar_directory.relative_to(project_root),
)
print(f"Official PDF files found: {len(official_calendar_paths)}")

for calendar_path in official_calendar_paths:
    print(f"- {calendar_path.name}")

Historical source: data\raw\extracted\school_holidays_figshare_v1\school.holidays.rda
R objects found:
- 'school.holidays': 58,440 rows × 4 columns

Official calendar directory: data\raw\source_documents\school_calendars_official_2024_2026
Official PDF files found: 9
- act_school_terms_2024_2026.pdf
- nsw_school_terms_2024_2025.pdf
- nsw_school_terms_2026_eastern.pdf
- nt_school_terms_2024_2026.pdf
- qld_school_terms_2024_2026.pdf
- sa_school_terms_2024_2026.pdf
- tas_school_terms_2024_2026.pdf
- vic_school_terms_2024_2026.pdf
- wa_school_terms_2024_2026.pdf


**Select and preview the historical holiday table**

The RDA container includes the expected `school.holidays` object. A copy is assigned to a clearly named DataFrame so that subsequent preparation does not alter the object returned by the source reader.

The initial preview displays one complete city set for the first date, making the city–date record grain visible.

In [7]:
# Select the expected R object explicitly rather than relying on
# its position within the RDA container.
historical_holiday_object_name = "school.holidays"

if historical_holiday_object_name not in historical_holiday_objects:
    raise KeyError(
        "Expected R object was not found: "
        f"{historical_holiday_object_name}"
    )

# Work with an independent in-memory copy while preserving the
# source object returned by pyreadr.
historical_school_holidays_raw = historical_holiday_objects[
    historical_holiday_object_name
].copy()

# Define the complete source schema expected from the audited dataset.
required_historical_holiday_columns = [
    "City",
    "Date",
    "schoolhols",
    "school.hols",
]

missing_historical_holiday_columns = [
    column
    for column in required_historical_holiday_columns
    if column not in historical_school_holidays_raw.columns
]

if missing_historical_holiday_columns:
    raise ValueError(
        "Missing historical school-holiday columns: "
        f"{missing_historical_holiday_columns}"
    )

# Present the source data types before any standardisation is applied.
historical_holiday_column_check = pd.DataFrame(
    {
        "column_name": required_historical_holiday_columns,
        "present_in_source": True,
        "source_dtype": [
            str(historical_school_holidays_raw[column].dtype)
            for column in required_historical_holiday_columns
        ],
    }
)

display(historical_holiday_column_check)

# Display the first eight records because the source contains
# eight city observations for each calendar date.
display(
    historical_school_holidays_raw.loc[
        :,
        required_historical_holiday_columns,
    ].head(8)
)

print("Historical school-holiday column check: PASSED")

,column_name,present_in_source,source_dtype
0,City,True,str
1,Date,True,object
2,schoolhols,True,category
3,school.hols,True,category


,City,Date,schoolhols,school.hols
0,Adelaide,2004-01-01,1,4
1,Brisbane,2004-01-01,1,4
2,Canberra,2004-01-01,1,4
3,Darwin,2004-01-01,1,4
4,Hobart,2004-01-01,1,4
5,Melbourne,2004-01-01,1,4
6,Perth,2004-01-01,1,4
7,Sydney,2004-01-01,1,4


Historical school-holiday column check: PASSED


**Result**

The `school.holidays` object loaded successfully with 58,440 rows and 4 source columns. All expected fields are present, and the preview is consistent with the intended grain of one city–date record.

`Date` is currently stored as an object, while `schoolhols` and `school.hols` were imported as categorical fields from the original R object. Their data types will be standardised in Section 5.

All 9 expected official calendar PDFs are available. These documents remain unchanged as source evidence and will be used to structure the reviewed 2024–2026 holiday periods later.

No date conversion, missing-value treatment, or calendar-period extraction was performed in this section.

## 3. Prepare the Historical Match Table

### 3.1 Standardise Column Names and Data Types

The historical source uses mixed column-name conventions and stores several analytical fields as text. This step creates a separate working table, renames the attributes using consistent `snake_case`, and converts them to appropriate data types.

The raw DataFrame and original CSV remain unchanged.

| Source column | Prepared column | Purpose |
|---|---|---|
| `GameId` | `source_game_id` | Retains the original AFLStats identifier for traceability. |
| `Year` | `season_year` | Identifies the AFL season. |
| `Round` | `round_label` | Retains the published round description for later correction and standardisation. |
| `Date` | `match_date` | Provides the calendar date of the match. |
| `StartTime` | `start_time` | Provides the scheduled local start time. |
| `HomeTeam` | `home_team` | Identifies the home team. |
| `AwayTeam` | `away_team` | Identifies the away team. |
| `Venue` | `venue_name` | Identifies the match venue. |
| `Attendance` | `attendance` | Provides the prediction target as an integer count. |
| `HomeTeamScore` | `home_score` | Supports later time-safe team-form features. |
| `AwayTeamScore` | `away_score` | Supports later time-safe team-form features. |

Quarter-by-quarter score fields are not required because we uses final scores only when constructing lagged team-form features.

The historical match-day weather observations are also excluded from the prepared table. Comparable forecast values are not yet available in the current future-scoring workflow, so using observed match-day weather during model development could create a mismatch between training and prediction. The original weather fields remain available in the unchanged raw source for a later weather-enabled version.

**Create the working table and standardise column names**

The selected source fields are copied into a separate working DataFrame and renamed consistently. This step changes the table structure only; source values and data types are not yet modified.

In [8]:
# Define an ordered source-to-prepared column mapping so that the
# resulting schema is explicit and reproducible.
historical_column_mapping = {
    "GameId": "source_game_id",
    "Year": "season_year",
    "Round": "round_label",
    "Date": "match_date",
    "StartTime": "start_time",
    "HomeTeam": "home_team",
    "AwayTeam": "away_team",
    "Venue": "venue_name",
    "Attendance": "attendance",
    "HomeTeamScore": "home_score",
    "AwayTeamScore": "away_score",
}

# Select only the version 1 fields and create an independent working
# table, leaving the previously loaded raw DataFrame unchanged.
historical_matches_working = (
    historical_matches_raw
    .loc[:, list(historical_column_mapping)]
    .rename(columns=historical_column_mapping)
    .copy()
)

# Confirm that column selection did not unintentionally remove rows
# and that the prepared column order matches the defined schema.
expected_historical_columns = list(
    historical_column_mapping.values()
)

if len(historical_matches_working) != len(historical_matches_raw):
    raise RuntimeError(
        "Historical row count changed during column selection."
    )

if (
    historical_matches_working.columns.tolist()
    != expected_historical_columns
):
    raise RuntimeError(
        "The historical working schema does not match "
        "the defined column mapping."
    )

print(
    "Rows retained:",
    f"{len(historical_matches_working):,} "
    f"of {len(historical_matches_raw):,}",
)
print(
    "Columns retained:",
    f"{historical_matches_working.shape[1]} "
    f"of {historical_matches_raw.shape[1]}",
)
print("\nPrepared column order:")
print(historical_matches_working.columns.tolist())

Rows retained: 2,879 of 2,879
Columns retained: 11 of 22

Prepared column order:
['source_game_id', 'season_year', 'round_label', 'match_date', 'start_time', 'home_team', 'away_team', 'venue_name', 'attendance', 'home_score', 'away_score']


**Convert fields to consistent data types**

Dates and times are parsed using their known source formats. Attendance values are converted from comma-formatted text to integer counts, while season and final-score fields are stored as nullable integers.

Parsing is performed into temporary Series first. Any value that cannot be converted stops the workflow before the working table is updated, preventing silent data loss.

In [9]:
# Parse dates using the explicit source format rather than relying on
# automatic format inference.
parsed_match_date = pd.to_datetime(
    historical_matches_working["match_date"],
    format="%Y-%m-%d",
    errors="coerce",
)

# Parse the 12-hour source times and standardise them as portable
# 24-hour strings suitable for later CSV export and database loading.
parsed_start_time = (
    pd.to_datetime(
        historical_matches_working["start_time"],
        format="%I:%M %p",
        errors="coerce",
    )
    .dt.strftime("%H:%M:%S")
    .astype("string")
)

# Remove thousands separators before converting attendance values
# from source text into numeric counts.
parsed_attendance = pd.to_numeric(
    historical_matches_working["attendance"]
    .astype("string")
    .str.replace(",", "", regex=False)
    .str.strip(),
    errors="coerce",
)

# Convert the season and final scores explicitly so unexpected
# non-numeric source values can be detected before assignment.
parsed_season_year = pd.to_numeric(
    historical_matches_working["season_year"],
    errors="coerce",
)

parsed_home_score = pd.to_numeric(
    historical_matches_working["home_score"],
    errors="coerce",
)

parsed_away_score = pd.to_numeric(
    historical_matches_working["away_score"],
    errors="coerce",
)

# Count conversion failures across fields that must be populated for
# the historical match table.
conversion_failure_counts = {
    "match_date": int(parsed_match_date.isna().sum()),
    "start_time": int(parsed_start_time.isna().sum()),
    "season_year": int(parsed_season_year.isna().sum()),
    "attendance": int(parsed_attendance.isna().sum()),
    "home_score": int(parsed_home_score.isna().sum()),
    "away_score": int(parsed_away_score.isna().sum()),
}

failed_conversions = {
    column: count
    for column, count in conversion_failure_counts.items()
    if count > 0
}

if failed_conversions:
    raise ValueError(
        "Historical type conversion failures detected: "
        f"{failed_conversions}"
    )

# Assign the validated conversions to the working table.
historical_matches_working["match_date"] = parsed_match_date
historical_matches_working["start_time"] = parsed_start_time
historical_matches_working["season_year"] = (
    parsed_season_year.astype("Int64")
)
historical_matches_working["attendance"] = (
    parsed_attendance.astype("Int64")
)
historical_matches_working["home_score"] = (
    parsed_home_score.astype("Int64")
)
historical_matches_working["away_score"] = (
    parsed_away_score.astype("Int64")
)

# Apply pandas string types to identifiers, labels, teams, and venues
# without changing their source values at this stage.
historical_string_columns = [
    "source_game_id",
    "round_label",
    "home_team",
    "away_team",
    "venue_name",
]

for column in historical_string_columns:
    historical_matches_working[column] = (
        historical_matches_working[column].astype("string")
    )

# Summarise the resulting schema and show a compact conversion preview.
historical_dtype_summary = pd.DataFrame(
    {
        "column_name": historical_matches_working.columns,
        "prepared_dtype": [
            str(historical_matches_working[column].dtype)
            for column in historical_matches_working.columns
        ],
    }
)

display(historical_dtype_summary)

display(
    historical_matches_working.loc[
        :,
        [
            "source_game_id",
            "match_date",
            "start_time",
            "attendance",
            "home_score",
            "away_score",
        ],
    ].head(3)
)

print("Historical data-type conversion check: PASSED")

,column_name,prepared_dtype
0,source_game_id,string
1,season_year,Int64
2,round_label,string
3,match_date,datetime64[us]
4,start_time,string
5,home_team,string
6,away_team,string
7,venue_name,string
8,attendance,Int64
9,home_score,Int64


,source_game_id,match_date,start_time,attendance,home_score,away_score
0,2012R0101,2012-03-24,19:20:00,38203,37,100
1,2012R0102,2012-03-29,19:45:00,78285,81,125
2,2012R0103,2012-03-30,19:50:00,78466,137,115


Historical data-type conversion check: PASSED


**Result**

The historical working table retains all 2,879 source records and contains the 11 fields selected for version 1.

Column names now follow a consistent `snake_case` convention. Match dates are stored as datetime values, start times use a consistent 24-hour representation, and season, attendance, and final-score fields use nullable integer types.

The comma separators were removed from attendance values before numeric conversion. All required values were converted successfully, and no source records were removed or modified in the raw DataFrame.

Team and venue values still retain their source spellings and will be standardised in the next step.

### 3.2 Standardise Team and Venue Names

An exact comparison of the AFLStats and Squiggle source values identified one team-name difference and three venue-name differences that require explicit mapping.

| Entity | AFLStats value | Squiggle value | Canonical value |
|---|---|---|---|
| Team | `Brisbane` | `Brisbane Lions` | `Brisbane Lions` |
| Venue | `MCG` | `M.C.G.` | `MCG` |
| Venue | `SCG` | `S.C.G.` | `SCG` |
| Venue | `Barossa Oval` | `Barossa Park` | `Barossa Park` |

`Brisbane Lions` is selected because it matches the 2026 Squiggle team reference. Unpunctuated `MCG` and `SCG` values are retained as concise canonical names.

`Barossa Park` is selected because it is the venue name currently used by the AFL for the Lyndoch venue.  
Source: [AFL — Barossa Park](https://www.afl.com.au/venues/178)

Only confirmed differences are mapped. Fuzzy matching is not used, and historical venues absent from the 2026 snapshot are retained because they represent valid matches from earlier seasons.

**Build and apply the team mapping**

The Squiggle teams snapshot is used as the 18-team reference because its identifiers are also present in the 2026 games snapshot.

The single confirmed AFLStats alias, `Brisbane`, is converted to `Brisbane Lions`. Exact canonical names are then used to attach the corresponding team identifiers to the historical match table. Both readable names and numeric identifiers are retained.

In [10]:
# Build a compact team reference from the Squiggle snapshot.
# The source identifiers provide a shared key for historical matches
# and the later 2026 match table.
team_reference = (
    squiggle_teams_raw.loc[
        :,
        ["id", "name", "abbrev"],
    ]
    .rename(
        columns={
            "id": "team_id",
            "name": "team_name",
            "abbrev": "team_abbreviation",
        }
    )
    .copy()
)

# Standardise the reference-table types before using it for lookups.
team_reference["team_id"] = pd.to_numeric(
    team_reference["team_id"],
    errors="raise",
).astype("Int64")

team_reference["team_name"] = (
    team_reference["team_name"]
    .astype("string")
    .str.strip()
)

team_reference["team_abbreviation"] = (
    team_reference["team_abbreviation"]
    .astype("string")
    .str.strip()
)

team_reference = (
    team_reference
    .sort_values("team_id")
    .reset_index(drop=True)
)

# Confirm that both identifiers and canonical names are unique.
if team_reference["team_id"].duplicated().any():
    raise ValueError(
        "Duplicate team identifiers were found in the team reference."
    )

if team_reference["team_name"].duplicated().any():
    raise ValueError(
        "Duplicate canonical team names were found in the team reference."
    )

# Define only the confirmed historical team alias.
team_alias_lookup = {
    "Brisbane": "Brisbane Lions",
}

# Strip accidental surrounding whitespace and apply the same explicit
# alias rule to both team roles.
team_value_change_count = 0

for column in ["home_team", "away_team"]:
    original_values = (
        historical_matches_working[column]
        .astype("string")
        .str.strip()
    )

    standardised_values = original_values.replace(
        team_alias_lookup
    )

    team_value_change_count += int(
        (original_values != standardised_values)
        .fillna(False)
        .sum()
    )

    historical_matches_working[column] = standardised_values

# Validate that every historical team now belongs to the canonical
# 18-team reference before attaching identifiers.
historical_team_names = set(
    pd.concat(
        [
            historical_matches_working["home_team"],
            historical_matches_working["away_team"],
        ],
        ignore_index=True,
    ).dropna()
)

canonical_team_names = set(
    team_reference["team_name"].dropna()
)

unmapped_team_names = sorted(
    historical_team_names - canonical_team_names
)

if unmapped_team_names:
    raise ValueError(
        "Historical teams missing from the canonical reference: "
        f"{unmapped_team_names}"
    )

# Attach the shared team identifiers through exact canonical-name
# lookups while retaining the readable team-name columns.
team_id_lookup = team_reference.set_index(
    "team_name"
)["team_id"]

historical_matches_working["home_team_id"] = (
    historical_matches_working["home_team"]
    .map(team_id_lookup)
    .astype("Int64")
)

historical_matches_working["away_team_id"] = (
    historical_matches_working["away_team"]
    .map(team_id_lookup)
    .astype("Int64")
)

if historical_matches_working[
    ["home_team_id", "away_team_id"]
].isna().any().any():
    raise ValueError(
        "One or more historical team identifiers could not be assigned."
    )

print(f"Canonical teams: {len(team_reference)}")
print(
    "Historical team values updated:",
    f"{team_value_change_count:,}",
)
print("Unmapped historical teams: 0")

display(team_reference)

display(
    historical_matches_working.loc[
        :,
        [
            "source_game_id",
            "home_team_id",
            "home_team",
            "away_team_id",
            "away_team",
        ],
    ].head(3)
)

print("Historical team mapping check: PASSED")

Canonical teams: 18
Historical team values updated: 326
Unmapped historical teams: 0


,team_id,team_name,team_abbreviation
0,1,Adelaide,ADE
1,2,Brisbane Lions,BRI
2,3,Carlton,CAR
3,4,Collingwood,COL
4,5,Essendon,ESS
5,6,Fremantle,FRE
6,7,Geelong,GEE
7,8,Gold Coast,GCS
8,9,Greater Western Sydney,GWS
9,10,Hawthorn,HAW


,source_game_id,home_team_id,home_team,away_team_id,away_team
0,2012R0101,9,Greater Western Sydney,16,Sydney
1,2012R0102,14,Richmond,3,Carlton
2,2012R0103,10,Hawthorn,4,Collingwood


Historical team mapping check: PASSED


**Apply the confirmed venue aliases**

The same explicit venue-alias rules are defined for reuse across the historical and 2026 match tables.

Only `Barossa Oval` occurs as a non-canonical value in the historical source. The punctuated `M.C.G.` and `S.C.G.` values occur in the Squiggle snapshot and will be handled later using the same mapping.

In [11]:
# Define the confirmed venue aliases once so the same rules can be
# reused when the 2026 Squiggle snapshot is prepared.
venue_alias_lookup = {
    "Barossa Oval": "Barossa Park",
    "M.C.G.": "MCG",
    "S.C.G.": "SCG",
}

# Remove accidental surrounding whitespace before applying only the
# confirmed exact-name replacements.
original_venue_names = (
    historical_matches_working["venue_name"]
    .astype("string")
    .str.strip()
)

standardised_venue_names = original_venue_names.replace(
    venue_alias_lookup
)

venue_value_change_count = int(
    (original_venue_names != standardised_venue_names)
    .fillna(False)
    .sum()
)

historical_matches_working["venue_name"] = (
    standardised_venue_names
)

# Summarise the historical records changed by the alias mapping.
venue_mapping_summary = (
    pd.DataFrame(
        {
            "source_venue_name": original_venue_names,
            "canonical_venue_name": standardised_venue_names,
        }
    )
    .loc[
        lambda frame:
        frame["source_venue_name"]
        != frame["canonical_venue_name"]
    ]
    .value_counts(
        [
            "source_venue_name",
            "canonical_venue_name",
        ]
    )
    .rename("records_updated")
    .reset_index()
)

# Confirm that no non-canonical alias remains in the historical table.
remaining_historical_venue_aliases = sorted(
    set(historical_matches_working["venue_name"].dropna())
    & set(venue_alias_lookup)
)

if remaining_historical_venue_aliases:
    raise ValueError(
        "Non-canonical historical venue aliases remain: "
        f"{remaining_historical_venue_aliases}"
    )

print(
    "Historical venue values updated:",
    f"{venue_value_change_count:,}",
)
print(
    "Canonical historical venues:",
    historical_matches_working["venue_name"].nunique(),
)
print("Remaining historical venue aliases: 0")

display(venue_mapping_summary)

print("Historical venue alias check: PASSED")

Historical venue values updated: 2
Canonical historical venues: 27
Remaining historical venue aliases: 0


,source_venue_name,canonical_venue_name,records_updated
0,Barossa Oval,Barossa Park,2


Historical venue alias check: PASSED


**Build the venue reference and school-holiday join key**

The historical holiday dataset uses eight Australian capital-city labels. A venue reference is therefore used to retain each venue’s actual location while assigning the corresponding calendar key required for the later school-holiday join.

For regional Australian venues, `school_holiday_city` identifies the capital-city calendar for the same state or territory. It does not imply that the venue is physically located in that capital city.

The two international venues are retained, but no Australian school-holiday key is assigned to them.

In [12]:
# Define the canonical venue locations and the corresponding city key
# available in the historical school-holiday dataset.
venue_reference_records = [
    ("Adelaide Oval", "Adelaide", "SA", "AU", "Adelaide"),
    ("Barossa Park", "Lyndoch", "SA", "AU", "Adelaide"),
    ("Bellerive Oval", "Hobart", "TAS", "AU", "Hobart"),
    ("Blacktown", "Blacktown", "NSW", "AU", "Sydney"),
    ("Carrara", "Gold Coast", "QLD", "AU", "Brisbane"),
    ("Cazaly's Stadium", "Cairns", "QLD", "AU", "Brisbane"),
    ("Docklands", "Melbourne", "VIC", "AU", "Melbourne"),
    ("Eureka Stadium", "Ballarat", "VIC", "AU", "Melbourne"),
    ("Football Park", "Adelaide", "SA", "AU", "Adelaide"),
    ("Gabba", "Brisbane", "QLD", "AU", "Brisbane"),
    ("Hands Oval", "Bunbury", "WA", "AU", "Perth"),
    ("Jiangwan Stadium", "Shanghai", pd.NA, "CN", pd.NA),
    ("Kardinia Park", "Geelong", "VIC", "AU", "Melbourne"),
    ("MCG", "Melbourne", "VIC", "AU", "Melbourne"),
    ("Manuka Oval", "Canberra", "ACT", "AU", "Canberra"),
    ("Marrara Oval", "Darwin", "NT", "AU", "Darwin"),
    ("Norwood Oval", "Adelaide", "SA", "AU", "Adelaide"),
    ("Perth Stadium", "Perth", "WA", "AU", "Perth"),
    ("Riverway Stadium", "Townsville", "QLD", "AU", "Brisbane"),
    ("SCG", "Sydney", "NSW", "AU", "Sydney"),
    ("Stadium Australia", "Sydney", "NSW", "AU", "Sydney"),
    ("Subiaco", "Perth", "WA", "AU", "Perth"),
    ("Summit Sports Park", "Mount Barker", "SA", "AU", "Adelaide"),
    ("Sydney Showground", "Sydney", "NSW", "AU", "Sydney"),
    ("Traeger Park", "Alice Springs", "NT", "AU", "Darwin"),
    ("Wellington", "Wellington", pd.NA, "NZ", pd.NA),
    ("York Park", "Launceston", "TAS", "AU", "Hobart"),
]

venue_reference = pd.DataFrame(
    venue_reference_records,
    columns=[
        "venue_name",
        "venue_city",
        "state_code",
        "country_code",
        "school_holiday_city",
    ],
).astype(
    {
        "venue_name": "string",
        "venue_city": "string",
        "state_code": "string",
        "country_code": "string",
        "school_holiday_city": "string",
    }
)

venue_reference = (
    venue_reference
    .sort_values("venue_name")
    .reset_index(drop=True)
)

# Require one reference record per canonical venue.
if venue_reference["venue_name"].duplicated().any():
    raise ValueError(
        "Duplicate canonical venue names were found "
        "in the venue reference."
    )

# Confirm that every Australian venue has a calendar join key.
australian_venues_missing_calendar_key = (
    venue_reference.loc[
        (venue_reference["country_code"] == "AU")
        & venue_reference["school_holiday_city"].isna()
    ]
)

if not australian_venues_missing_calendar_key.empty:
    raise ValueError(
        "Australian venues are missing school-holiday city keys."
    )

# Validate coverage of all historical canonical venue names.
historical_venue_names = set(
    historical_matches_working["venue_name"].dropna()
)

reference_venue_names = set(
    venue_reference["venue_name"].dropna()
)

historical_venues_missing_reference = sorted(
    historical_venue_names - reference_venue_names
)

if historical_venues_missing_reference:
    raise ValueError(
        "Historical venues missing from the venue reference: "
        f"{historical_venues_missing_reference}"
    )

# Standardise a temporary copy of the Squiggle venue names solely to
# confirm that the same reference will cover the 2026 snapshot.
squiggle_venue_names_for_coverage = (
    squiggle_games_raw["venue"]
    .astype("string")
    .str.strip()
    .replace(venue_alias_lookup)
)

squiggle_venues_missing_reference = sorted(
    set(squiggle_venue_names_for_coverage.dropna())
    - reference_venue_names
)

if squiggle_venues_missing_reference:
    raise ValueError(
        "Squiggle venues missing from the venue reference: "
        f"{squiggle_venues_missing_reference}"
    )

international_venue_reference = venue_reference.loc[
    venue_reference["country_code"] != "AU"
]

print(f"Venue reference records: {len(venue_reference)}")
print(
    "Historical venues missing reference:",
    len(historical_venues_missing_reference),
)
print(
    "Squiggle venues missing reference:",
    len(squiggle_venues_missing_reference),
)
print(
    "International venues without Australian calendar keys:",
    len(international_venue_reference),
)

display(venue_reference.head(8))

print("International venue records:")
display(international_venue_reference)

print("Venue reference coverage check: PASSED")

Venue reference records: 27
Historical venues missing reference: 0
Squiggle venues missing reference: 0
International venues without Australian calendar keys: 2


,venue_name,venue_city,state_code,country_code,school_holiday_city
0,Adelaide Oval,Adelaide,SA,AU,Adelaide
1,Barossa Park,Lyndoch,SA,AU,Adelaide
2,Bellerive Oval,Hobart,TAS,AU,Hobart
3,Blacktown,Blacktown,NSW,AU,Sydney
4,Carrara,Gold Coast,QLD,AU,Brisbane
5,Cazaly's Stadium,Cairns,QLD,AU,Brisbane
6,Docklands,Melbourne,VIC,AU,Melbourne
7,Eureka Stadium,Ballarat,VIC,AU,Melbourne


International venue records:


,venue_name,venue_city,state_code,country_code,school_holiday_city
11,Jiangwan Stadium,Shanghai,<NA>,CN,<NA>
25,Wellington,Wellington,<NA>,NZ,<NA>


Venue reference coverage check: PASSED


**Attach the venue reference to historical matches**

The venue reference is joined to the historical working table using a validated many-to-one relationship: many matches may share one canonical venue, while each venue must have exactly one reference record.

The join must preserve the historical row count. International matches remain in the table with a valid venue and country, but without an Australian school-holiday city key.

In [13]:
# Record the pre-join row count so an accidental one-to-many expansion
# can be detected immediately.
historical_rows_before_venue_join = len(
    historical_matches_working
)

# Attach venue location and school-calendar keys through the canonical
# venue name. The validation requires one reference row per venue.
historical_matches_working = (
    historical_matches_working
    .merge(
        venue_reference,
        how="left",
        on="venue_name",
        validate="many_to_one",
    )
)

# Confirm that the reference join preserved one row per source match.
if (
    len(historical_matches_working)
    != historical_rows_before_venue_join
):
    raise RuntimeError(
        "Historical row count changed during the venue-reference join."
    )

# Every match must receive a venue city and country code. State and
# school-holiday keys may remain missing only for international venues.
missing_core_venue_reference = (
    historical_matches_working[
        ["venue_city", "country_code"]
    ]
    .isna()
    .any(axis=1)
)

if missing_core_venue_reference.any():
    raise ValueError(
        "One or more historical matches did not receive "
        "a valid venue reference."
    )

australian_matches_missing_calendar_key = (
    (historical_matches_working["country_code"] == "AU")
    & (
        historical_matches_working[
            ["state_code", "school_holiday_city"]
        ]
        .isna()
        .any(axis=1)
    )
)

if australian_matches_missing_calendar_key.any():
    raise ValueError(
        "Australian matches are missing state or "
        "school-holiday city values."
    )

# Reorder the working table so identifiers, match details, location
# fields, target, and result fields appear in a logical sequence.
historical_matches_working = (
    historical_matches_working.loc[
        :,
        [
            "source_game_id",
            "season_year",
            "round_label",
            "match_date",
            "start_time",
            "home_team_id",
            "home_team",
            "away_team_id",
            "away_team",
            "venue_name",
            "venue_city",
            "state_code",
            "country_code",
            "school_holiday_city",
            "attendance",
            "home_score",
            "away_score",
        ],
    ]
)

# Summarise the retained international matches separately because
# Australian school-holiday features do not apply to them.
international_match_summary = (
    historical_matches_working.loc[
        historical_matches_working["country_code"] != "AU"
    ]
    .groupby(
        [
            "venue_name",
            "venue_city",
            "country_code",
        ],
        dropna=False,
    )
    .size()
    .rename("match_count")
    .reset_index()
)

print(
    "Historical rows before venue join:",
    f"{historical_rows_before_venue_join:,}",
)
print(
    "Historical rows after venue join:",
    f"{len(historical_matches_working):,}",
)
print(
    "International historical matches:",
    f"{int(international_match_summary['match_count'].sum()):,}",
)

display(
    historical_matches_working.loc[
        :,
        [
            "source_game_id",
            "venue_name",
            "venue_city",
            "state_code",
            "country_code",
            "school_holiday_city",
        ],
    ].head(5)
)

print("International match summary:")
display(international_match_summary)

print("Historical venue-reference join check: PASSED")

Historical rows before venue join: 2,879
Historical rows after venue join: 2,879
International historical matches: 6


,source_game_id,venue_name,venue_city,state_code,country_code,school_holiday_city
0,2012R0101,Stadium Australia,Sydney,NSW,AU,Sydney
1,2012R0102,MCG,Melbourne,VIC,AU,Melbourne
2,2012R0103,MCG,Melbourne,VIC,AU,Melbourne
3,2012R0104,MCG,Melbourne,VIC,AU,Melbourne
4,2012R0105,Carrara,Gold Coast,QLD,AU,Brisbane


International match summary:


,venue_name,venue_city,country_code,match_count
0,Jiangwan Stadium,Shanghai,CN,3
1,Wellington,Wellington,NZ,3


Historical venue-reference join check: PASSED


**Result**

All historical team values now align with the 18-team Squiggle reference. A total of 326 home-team or away-team values were updated from the AFLStats alias `Brisbane` to the canonical name `Brisbane Lions`. Every historical match received valid home and away team identifiers.

The venue reference contains 27 canonical venues. Two historical records were updated from `Barossa Oval` to `Barossa Park`, while the same reference also covers the punctuated `M.C.G.` and `S.C.G.` values found in the 2026 Squiggle snapshot.

The validated many-to-one venue join preserved all 2,879 historical match records. Every Australian venue received a state or territory code and a school-holiday city key.

Six international matches were retained: three at Jiangwan Stadium in Shanghai and three in Wellington. These records have valid venue and country information but intentionally have no Australian school-holiday key.

All mappings used explicit confirmed values; no fuzzy matching was applied.

### 3.3 Apply the Reviewed 2024 Round Correlations

The official 2024 AFL fixture contains four Opening Round matches played from 7 to 9 March, followed by Round 1 from 14 to 17 March.

The historical source assigns both weekends to `Opening Round` and subsequently numbers the remaining home-and-away rounds one round lower, ending with `Round 23` rather than `Round 24`.

The following transparent corrections are therefore applied:

- retain the four official Opening Round matches;
- relabel the nine matches played from 14 to 17 March as `Round 1`;
- shift source labels `Round 1`–`Round 23` to `Round 2`–`Round 24`;
- leave all finals labels unchanged;
- preserve the original value in `source_round_label`.

The raw CSV and `source_game_id` values remain unchanged.

Source: [AFL — 2024 fixture announcement](https://www.afl.com.au/news/1064233)

In [14]:
# Work on a copy so that validation failures cannot partially update
# the prepared historical match table.
historical_rounds_candidate = historical_matches_working.copy()
original_row_count = len(historical_rounds_candidate)

# Preserve the source-provided round label before applying corrections.
round_label_position = historical_rounds_candidate.columns.get_loc("round_label")
historical_rounds_candidate.insert(
    round_label_position,
    "source_round_label",
    historical_rounds_candidate["round_label"],
)

# Identify the nine matches incorrectly included in Opening Round.
opening_round_to_round_1_mask = (
    historical_rounds_candidate["season_year"].eq(2024)
    & historical_rounds_candidate["source_round_label"].eq("Opening Round")
    & historical_rounds_candidate["match_date"].between(
        pd.Timestamp("2024-03-14"),
        pd.Timestamp("2024-03-17"),
    )
)

# Identify the remaining 2024 home-and-away rounds whose numeric labels
# are one round lower than the official fixture numbering.
regular_round_shift_mask = (
    historical_rounds_candidate["season_year"].eq(2024)
    & historical_rounds_candidate["source_round_label"].str.fullmatch(
        r"Round \d+",
        na=False,
    )
)

# Correct the nine Round 1 matches.
historical_rounds_candidate.loc[
    opening_round_to_round_1_mask,
    "round_label",
] = "Round 1"

# Shift source Round 1–23 labels to official Round 2–24 labels.
source_round_numbers = (
    historical_rounds_candidate.loc[
        regular_round_shift_mask,
        "source_round_label",
    ]
    .str.extract(r"(\d+)", expand=False)
    .astype("Int64")
)

historical_rounds_candidate.loc[
    regular_round_shift_mask,
    "round_label",
] = "Round " + (source_round_numbers + 1).astype("string")

# Record every changed row using transparent source and corrected values.
round_value_changed_mask = (
    historical_rounds_candidate["round_label"]
    != historical_rounds_candidate["source_round_label"]
)

round_correction_log = (
    historical_rounds_candidate.loc[
        round_value_changed_mask,
        [
            "source_game_id",
            "match_date",
            "home_team",
            "away_team",
            "source_round_label",
            "round_label",
        ],
    ]
    .rename(columns={"round_label": "corrected_round_label"})
    .reset_index(drop=True)
)

round_correction_log["correction_reason"] = (
    "Align the source label with the official 2024 AFL fixture numbering."
)

# Calculate the expected validation counts.
opening_round_corrections = int(opening_round_to_round_1_mask.sum())
numeric_round_corrections = int(regular_round_shift_mask.sum())
total_round_corrections = len(round_correction_log)

opening_round_2024_count = int(
    (
        historical_rounds_candidate["season_year"].eq(2024)
        & historical_rounds_candidate["round_label"].eq("Opening Round")
    ).sum()
)

round_1_2024_count = int(
    (
        historical_rounds_candidate["season_year"].eq(2024)
        & historical_rounds_candidate["round_label"].eq("Round 1")
    ).sum()
)

round_24_2024_count = int(
    (
        historical_rounds_candidate["season_year"].eq(2024)
        & historical_rounds_candidate["round_label"].eq("Round 24")
    ).sum()
)

# Validate the correction scope and final round structure.
if len(historical_rounds_candidate) != original_row_count:
    raise ValueError("The round correction changed the historical row count.")

if opening_round_corrections != 9:
    raise ValueError(
        "The expected nine Opening Round records were not identified."
    )

if numeric_round_corrections != 194:
    raise ValueError(
        "The expected 194 numeric round records were not identified."
    )

if total_round_corrections != 203:
    raise ValueError(
        "The total number of corrected 2024 records is not 203."
    )

if opening_round_2024_count != 4:
    raise ValueError(
        "The corrected 2024 Opening Round does not contain four matches."
    )

if round_1_2024_count != 9:
    raise ValueError(
        "The corrected 2024 Round 1 does not contain nine matches."
    )

if round_24_2024_count != 9:
    raise ValueError(
        "The corrected 2024 Round 24 does not contain nine matches."
    )

# Update the working table only after every validation has passed.
historical_matches_working = historical_rounds_candidate

round_correction_summary = pd.DataFrame(
    {
        "correction_group": [
            "Opening Round to Round 1",
            "Round 1–23 to Round 2–24",
        ],
        "records_corrected": [
            opening_round_corrections,
            numeric_round_corrections,
        ],
    }
)

display(round_correction_summary)

print(f"Total records corrected: {total_round_corrections:,}")
print(f"2024 Opening Round matches: {opening_round_2024_count}")
print(f"2024 Round 1 matches: {round_1_2024_count}")
print(f"2024 Round 24 matches: {round_24_2024_count}")
print("Reviewed 2024 round-correction check: PASSED")

,correction_group,records_corrected
0,Opening Round to Round 1,9
1,Round 1–23 to Round 2–24,194


Total records corrected: 203
2024 Opening Round matches: 4
2024 Round 1 matches: 9
2024 Round 24 matches: 9
Reviewed 2024 round-correction check: PASSED


**Result**

The reviewed correction rules updated 203 round labels for the 2024 season:

- nine matches were reclassified from `Opening Round` to `Round 1`;
- 194 regular-season matches were shifted from source labels `Round 1`–`Round 23` to official labels `Round 2`–`Round 24`.

The corrected data now contains four Opening Round matches, nine Round 1 matches, and nine Round 24 matches. Finals labels were left unchanged.

The original values remain available in `source_round_label`, while `round_label` contains the corrected values used in subsequent analysis. No raw files, source identifiers, or match records were modified or removed.

### 3.4 Create and Validate the Match Identifier

The source-provided `GameId` cannot be used as the prepared table's unique identifier because three values are each assigned to two different matches.

A deterministic `match_id` is therefore constructed from:

- the match date;
- the canonical home-team identifier;
- the canonical away-team identifier.

The identifier uses the readable format `YYYYMMDD_H##_A##`. For example, `20120324_H09_A16` represents the match played on 24 March 2012 between home team 9 and away team 16.

The home and away roles are retained because reversing the teams represents a different match. Venue, score, attendance, and round label are excluded because they describe the match but are not required to identify it.

The original identifier remains available as `source_game_id` for source traceability.

In [15]:
# Confirm that the three match-identity components are complete and unique.
match_identity_columns = [
    "match_date",
    "home_team_id",
    "away_team_id",
]

assert (
    historical_matches_working[
        match_identity_columns
    ].notna().all().all()
), "Some matches have missing identifier components."

assert not historical_matches_working.duplicated(
    subset=match_identity_columns
).any(), (
    "Match date, home-team ID, and away-team ID "
    "do not form a unique identity."
)

# Create a readable project identifier while retaining source_game_id.
historical_matches_prepared = historical_matches_working.copy()

match_id = (
    historical_matches_prepared["match_date"].dt.strftime("%Y%m%d")
    + "_H"
    + historical_matches_prepared["home_team_id"]
    .astype("int64")
    .astype("string")
    .str.zfill(2)
    + "_A"
    + historical_matches_prepared["away_team_id"]
    .astype("int64")
    .astype("string")
    .str.zfill(2)
)

historical_matches_prepared.insert(
    0,
    "match_id",
    match_id,
)

# Validate the new identifier and retain the reviewed source-ID collisions.
duplicate_match_id_rows = int(
    historical_matches_prepared["match_id"]
    .duplicated(keep=False)
    .sum()
)

source_id_collision_mask = (
    historical_matches_prepared["source_game_id"]
    .duplicated(keep=False)
)

duplicate_source_id_values = int(
    historical_matches_prepared.loc[
        source_id_collision_mask,
        "source_game_id",
    ].nunique()
)

assert duplicate_match_id_rows == 0, (
    "The prepared match_id contains duplicate values."
)

assert duplicate_source_id_values == 3, (
    "The reviewed source_game_id collision count has changed."
)

# Show how the new identifier separates the six source collision records.
source_id_collision_review = (
    historical_matches_prepared.loc[
        source_id_collision_mask,
        [
            "match_id",
            "source_game_id",
            "match_date",
            "home_team",
            "away_team",
            "round_label",
        ],
    ]
    .sort_values(["source_game_id", "match_date"])
    .reset_index(drop=True)
)

identifier_validation_summary = pd.DataFrame(
    {
        "check": [
            "Historical rows",
            "Unique match_id values",
            "Duplicate match_id rows",
            "Duplicated source_game_id values retained",
        ],
        "value": [
            len(historical_matches_prepared),
            historical_matches_prepared["match_id"].nunique(),
            duplicate_match_id_rows,
            duplicate_source_id_values,
        ],
        "status": [
            "PASSED",
            "PASSED",
            "PASSED",
            "EXPECTED",
        ],
    }
)

display(source_id_collision_review)
display(identifier_validation_summary)

print("Historical match identifier check: PASSED")

,match_id,source_game_id,match_date,home_team,away_team,round_label
0,20240308_H02_A03,2024OR01,2024-03-08,Brisbane Lions,Carlton,Opening Round
1,20240314_H03_A14,2024OR01,2024-03-14,Carlton,Richmond,Round 1
2,20240309_H08_A14,2024OR02,2024-03-09,Gold Coast,Richmond,Opening Round
3,20240315_H04_A16,2024OR02,2024-03-15,Collingwood,Sydney,Round 1
4,20240309_H09_A04,2024OR03,2024-03-09,Greater Western Sydney,Collingwood,Opening Round
5,20240316_H05_A10,2024OR03,2024-03-16,Essendon,Hawthorn,Round 1


,check,value,status
0,Historical rows,2879,PASSED
1,Unique match_id values,2879,PASSED
2,Duplicate match_id rows,0,PASSED
3,Duplicated source_game_id values retained,3,EXPECTED


Historical match identifier check: PASSED


**Result**

A unique and readable `match_id` was created for all 2,879 historical matches using the match date, canonical home-team ID, and canonical away-team ID.

No missing or duplicate project identifiers were found. The three duplicated `source_game_id` values affect six different matches, but each of these records now has a distinct `match_id`.

The source identifiers remain unchanged for traceability, while `match_id` will be used as the unique identifier for the prepared historical match table.  
  
The completed Section 3 output is stored in `historical_matches_prepared`. The supporting team and venue reference tables remain available as `team_reference` and `venue_reference`, while the reviewed 2024 round changes are retained in `round_correction_log`.

## 4. Prepare the 2026 Squiggle Match Snapshot

### 4.1 Select the Required Match and Fixture Fields

The Squiggle games response contains 26 source fields. This section retains only the fields required for fixture identification, scheduling, team and venue mapping, match status, final scores, and source traceability.

The selected field groups are:

- source identity and snapshot date;
- season and round information;
- source, local, and Unix time values;
- home and away team identifiers and names;
- venue;
- completion percentage and finals classification;
- final home and away scores;
- source update timestamp.

The source field `complete` represents match completion as a percentage. The source field `is_final` identifies the type of finals match and must not be interpreted as a completed-match flag. They are therefore renamed to `completion_percentage` and `finals_type_code`.

Goals, behinds, winner fields, and display-only status text are excluded because they are either outside the v1 scope or can be derived from retained fields.

All records are retained at this stage, including finals placeholders whose participating teams were not yet known on the snapshot date.

Source documentation: [Squiggle API](https://api.squiggle.com.au/)

In [16]:
# Define an explicit source-to-prepared field selection so that the
# 2026 snapshot contains only fields required by the v1 workflow.
squiggle_match_field_map = {
    "id": "source_game_id",
    "year": "season_year",
    "round": "round_number",
    "roundname": "round_label",
    "date": "source_datetime",
    "localtime": "source_local_datetime",
    "tz": "source_utc_offset",
    "unixtime": "source_unix_time",
    "hteamid": "source_home_team_id",
    "hteam": "source_home_team",
    "ateamid": "source_away_team_id",
    "ateam": "source_away_team",
    "venue": "source_venue_name",
    "complete": "completion_percentage",
    "is_final": "finals_type_code",
    "hscore": "home_score",
    "ascore": "away_score",
    "updated": "source_updated_at",
}

# Confirm that the fixed API snapshot contains every required source field.
missing_squiggle_fields = sorted(
    set(squiggle_match_field_map)
    - set(squiggle_games_raw.columns)
)

if missing_squiggle_fields:
    raise KeyError(
        "Missing required Squiggle game fields: "
        f"{missing_squiggle_fields}"
    )

source_squiggle_row_count = len(squiggle_games_raw)

# Select and rename the required fields without changing their values.
squiggle_matches_working = (
    squiggle_games_raw[
        list(squiggle_match_field_map)
    ]
    .rename(columns=squiggle_match_field_map)
    .copy()
)

# Record the date represented by the fixed API snapshot filename.
squiggle_matches_working.insert(
    1,
    "snapshot_date",
    pd.Timestamp("2026-08-17"),
)

# Confirm that field selection has not removed or duplicated any records.
if len(squiggle_matches_working) != source_squiggle_row_count:
    raise ValueError(
        "The Squiggle field-selection step changed the source row count."
    )

expected_selected_column_count = len(squiggle_match_field_map) + 1

if (
    squiggle_matches_working.shape[1]
    != expected_selected_column_count
):
    raise ValueError(
        "The prepared Squiggle table does not contain the expected columns."
    )

# Count unresolved records without treating expected finals placeholders
# as data-quality failures.
unassigned_team_records = int(
    squiggle_matches_working[
        ["source_home_team_id", "source_away_team_id"]
    ]
    .isna()
    .any(axis=1)
    .sum()
)

squiggle_selection_preview_columns = [
    "source_game_id",
    "snapshot_date",
    "season_year",
    "round_number",
    "round_label",
    "source_local_datetime",
    "source_home_team",
    "source_away_team",
    "source_venue_name",
    "completion_percentage",
    "finals_type_code",
]

display(
    squiggle_matches_working[
        squiggle_selection_preview_columns
    ].head(3)
)

print(f"Source records retained: {len(squiggle_matches_working):,}")
print(
    "Source fields retained: "
    f"{len(squiggle_match_field_map)} "
    f"of {squiggle_games_raw.shape[1]}"
)
print(
    "Prepared columns including snapshot_date: "
    f"{squiggle_matches_working.shape[1]}"
)
print(
    "Records with teams not yet assigned: "
    f"{unassigned_team_records}"
)
print("Squiggle match-field selection check: PASSED")

,source_game_id,snapshot_date,season_year,round_number,round_label,source_local_datetime,source_home_team,source_away_team,source_venue_name,completion_percentage,finals_type_code
0,38494,2026-08-17,2026,0,Opening Round,2026-03-05 19:30:00,Sydney,Carlton,S.C.G.,100,0
1,38495,2026-08-17,2026,0,Opening Round,2026-03-06 19:05:00,Gold Coast,Geelong,Carrara,100,0
2,38496,2026-08-17,2026,0,Opening Round,2026-03-07 16:15:00,Greater Western Sydney,Hawthorn,Sydney Showground,100,0


Source records retained: 218
Source fields retained: 18 of 26
Prepared columns including snapshot_date: 19
Records with teams not yet assigned: 11
Squiggle match-field selection check: PASSED


**Result**

All 218 records from the fixed Squiggle games snapshot were retained. Eighteen of the 26 source fields were selected, and `snapshot_date` was added to record when the API data was collected.

Eleven records do not yet have assigned home and away teams. These are unresolved finals placeholders in the source snapshot and have therefore been retained rather than treated as missing-data errors.

No data types, names, venues, match statuses, or scores were modified during this field-selection step.

### 4.2 Apply Team and Venue Name Mappings

The 2026 Squiggle records are aligned with the same team and venue references used for the historical match table.

For teams, the Squiggle team identifier is used as the mapping key. This is more reliable than matching names alone. Canonical team identifiers and names are added while the original Squiggle values remain available in the `source_` columns.

Records representing unresolved finals fixtures are expected to have missing team identifiers and names. These records are retained, while all assigned teams must match the 18-team reference table.

For venues, the reviewed aliases from Section 3.2 are reused:

- `M.C.G.` → `MCG`
- `S.C.G.` → `SCG`
- `Barossa Oval` → `Barossa Park`

The standardised venue is then connected to `venue_reference` to add its city, state or territory, country, and school-holiday calendar city.

No fuzzy matching or automatic entity-resolution method is used. Any assigned team or venue not covered by the reviewed references causes validation to fail.

In [17]:
# Create canonical team fields while retaining the original API values.
squiggle_mapped = squiggle_matches_working.copy()
input_row_count = len(squiggle_mapped)

assigned_team_mask = (
    squiggle_mapped["source_home_team_id"].notna()
    & squiggle_mapped["source_away_team_id"].notna()
)

placeholder_team_mask = (
    squiggle_mapped["source_home_team_id"].isna()
    & squiggle_mapped["source_away_team_id"].isna()
)

# Each record must contain either two assigned teams or no assigned teams.
assert (assigned_team_mask | placeholder_team_mask).all(), (
    "Some Squiggle records contain only one assigned team."
)

assert squiggle_mapped.loc[
    assigned_team_mask,
    ["source_home_team", "source_away_team"],
].notna().all().all(), (
    "Some assigned team identifiers have missing source names."
)

team_name_by_id = (
    team_reference
    .set_index("team_id")["team_name"]
)

for side in ["home", "away"]:
    source_id_column = f"source_{side}_team_id"
    canonical_id_column = f"{side}_team_id"
    canonical_name_column = f"{side}_team"

    squiggle_mapped[canonical_id_column] = pd.to_numeric(
        squiggle_mapped[source_id_column],
        errors="coerce",
    ).astype("Int64")

    squiggle_mapped[canonical_name_column] = (
        squiggle_mapped[canonical_id_column]
        .map(team_name_by_id)
        .astype("string")
    )

# Assigned IDs must map to names that agree with the source values.
unmapped_team_records = int(
    squiggle_mapped.loc[
        assigned_team_mask,
        ["home_team", "away_team"],
    ].isna().any(axis=1).sum()
)

assert unmapped_team_records == 0, (
    "Some assigned team IDs are missing from team_reference."
)

team_name_mismatch_count = sum(
    int(
        squiggle_mapped.loc[
            assigned_team_mask,
            f"source_{side}_team",
        ]
        .astype("string")
        .ne(
            squiggle_mapped.loc[
                assigned_team_mask,
                f"{side}_team",
            ]
        )
        .sum()
    )
    for side in ["home", "away"]
)

assert team_name_mismatch_count == 0, (
    "Some Squiggle team IDs disagree with their source names."
)

# Apply the reviewed venue aliases from Section 3.2.
squiggle_mapped["venue_name"] = (
    squiggle_mapped["source_venue_name"]
    .replace(venue_alias_lookup)
    .astype("string")
)

venue_changed_mask = (
    squiggle_mapped["source_venue_name"]
    .astype("string")
    .ne(squiggle_mapped["venue_name"])
)

venue_mapping_review = (
    squiggle_mapped.loc[
        venue_changed_mask,
        ["source_venue_name", "venue_name"],
    ]
    .groupby(
        ["source_venue_name", "venue_name"]
    )
    .size()
    .rename("records_updated")
    .reset_index()
)

# Add the reviewed venue location and school-calendar fields.
venue_columns = [
    "venue_name",
    "venue_city",
    "state_code",
    "country_code",
    "school_holiday_city",
]

squiggle_mapped = squiggle_mapped.merge(
    venue_reference[venue_columns],
    on="venue_name",
    how="left",
    validate="many_to_one",
    indicator=True,
)

unmapped_venue_count = int(
    squiggle_mapped["_merge"].ne("both").sum()
)

squiggle_mapped = squiggle_mapped.drop(columns="_merge")

assert len(squiggle_mapped) == input_row_count, (
    "Venue mapping changed the Squiggle row count."
)

assert unmapped_venue_count == 0, (
    "Some Squiggle venues are missing from venue_reference."
)

squiggle_matches_working = squiggle_mapped

mapping_summary = pd.DataFrame(
    {
        "check": [
            "Matches with assigned teams",
            "Finals placeholders retained",
            "Team ID-name mismatches",
            "Venue values standardised",
            "Unmapped venues",
        ],
        "value": [
            int(assigned_team_mask.sum()),
            int(placeholder_team_mask.sum()),
            team_name_mismatch_count,
            int(venue_changed_mask.sum()),
            unmapped_venue_count,
        ],
        "status": [
            "PASSED",
            "EXPECTED",
            "PASSED",
            "EXPECTED",
            "PASSED",
        ],
    }
)

display(venue_mapping_review)
display(mapping_summary)

print("Squiggle team and venue mapping check: PASSED")

,source_venue_name,venue_name,records_updated
0,M.C.G.,MCG,57
1,S.C.G.,SCG,11


,check,value,status
0,Matches with assigned teams,207,PASSED
1,Finals placeholders retained,11,EXPECTED
2,Team ID-name mismatches,0,PASSED
3,Venue values standardised,68,EXPECTED
4,Unmapped venues,0,PASSED


Squiggle team and venue mapping check: PASSED


**Result**

All 207 matches with assigned teams were successfully aligned with the 18-team reference table. No inconsistencies were found between the Squiggle team identifiers and team names.

Eleven unresolved finals placeholders were retained with missing team assignments, as expected for the fixed snapshot date.

Venue standardisation updated 68 records: 57 occurrences of `M.C.G.` were mapped to `MCG`, and 11 occurrences of `S.C.G.` were mapped to `SCG`. All 2026 venue values were successfully connected to `venue_reference`, with no unmapped venues or changes to the source row count.

The original Squiggle team and venue values remain available in the corresponding `source_` columns.

### 4.3 Standardise Match Dates and Status

Squiggle provides several representations of match time. They serve different purposes and must not be treated as interchangeable.

- `source_datetime` and `source_utc_offset` describe the scheduled instant using the API's reference offset;
- `source_unix_time` provides the same instant independently of display timezone;
- `source_local_datetime` represents the scheduled time at the match venue.

The venue-local value is used to create `match_date` and `start_time`, which will support weekday, time-of-day, and calendar joins. The Unix timestamp is converted to `match_datetime_utc` for unambiguous chronological comparisons.

Match completion is standardised as:

- `0` → `not_started`;
- `1`–`99` → `in_progress`;
- `100` → `completed`.

The source snapshot stores zero scores for matches that have not started. These values are retained in `source_home_score` and `source_away_score`, while the prepared score fields are set to missing until a match is completed.

This section standardises completion status only. The distinction between scheduled fixtures and unresolved finals placeholders is made in Section 4.4.

In [18]:
# Standardise the selected numeric, datetime, and text fields.
squiggle_prepared = squiggle_matches_working.copy()
input_row_count = len(squiggle_prepared)

integer_columns = [
    "source_game_id",
    "season_year",
    "round_number",
    "source_unix_time",
    "source_home_team_id",
    "source_away_team_id",
    "home_team_id",
    "away_team_id",
    "completion_percentage",
    "finals_type_code",
    "home_score",
    "away_score",
]

squiggle_prepared[integer_columns] = (
    squiggle_prepared[integer_columns]
    .apply(pd.to_numeric, errors="raise")
    .astype("Int64")
)

datetime_columns = [
    "snapshot_date",
    "source_datetime",
    "source_local_datetime",
    "source_updated_at",
]

squiggle_prepared[datetime_columns] = (
    squiggle_prepared[datetime_columns]
    .apply(pd.to_datetime, errors="raise")
)

string_columns = [
    "round_label",
    "source_utc_offset",
    "source_home_team",
    "source_away_team",
    "source_venue_name",
    "home_team",
    "away_team",
    "venue_name",
    "venue_city",
    "state_code",
    "country_code",
    "school_holiday_city",
]

squiggle_prepared[string_columns] = (
    squiggle_prepared[string_columns].astype("string")
)

# Convert the Unix timestamp to UTC and verify it against the API
# datetime-plus-offset representation.
squiggle_prepared["match_datetime_utc"] = pd.to_datetime(
    squiggle_prepared["source_unix_time"].astype("int64"),
    unit="s",
    utc=True,
)

utc_from_source_offset = pd.to_datetime(
    squiggle_prepared["source_datetime"].dt.strftime(
        "%Y-%m-%d %H:%M:%S"
    )
    + squiggle_prepared["source_utc_offset"],
    utc=True,
)

utc_mismatch_count = int(
    utc_from_source_offset.ne(
        squiggle_prepared["match_datetime_utc"]
    ).sum()
)

# Use venue-local time for matchday calendar and time-of-day fields.
squiggle_prepared["match_date"] = (
    squiggle_prepared[
        "source_local_datetime"
    ].dt.normalize()
)

squiggle_prepared["start_time"] = (
    squiggle_prepared[
        "source_local_datetime"
    ]
    .dt.strftime("%H:%M:%S")
    .astype("string")
)

# Convert completion percentage into a readable status.
assert (
    squiggle_prepared["completion_percentage"].notna().all()
    and squiggle_prepared[
        "completion_percentage"
    ].between(0, 100).all()
), "completion_percentage contains missing or invalid values."

completion_status = pd.Series(
    "in_progress",
    index=squiggle_prepared.index,
    dtype="string",
)

completion_status.loc[
    squiggle_prepared["completion_percentage"].eq(0)
] = "not_started"

completion_status.loc[
    squiggle_prepared["completion_percentage"].eq(100)
] = "completed"

squiggle_prepared["completion_status"] = completion_status

# Preserve source scores and remove zero placeholders from the prepared
# score fields for matches that have not been completed.
squiggle_prepared = squiggle_prepared.rename(
    columns={
        "home_score": "source_home_score",
        "away_score": "source_away_score",
    }
)

completed_match_mask = (
    squiggle_prepared["completion_status"].eq("completed")
)

squiggle_prepared["home_score"] = (
    squiggle_prepared["source_home_score"]
    .where(completed_match_mask, pd.NA)
    .astype("Int64")
)

squiggle_prepared["away_score"] = (
    squiggle_prepared["source_away_score"]
    .where(completed_match_mask, pd.NA)
    .astype("Int64")
)

# Retain only the key checks needed at this preparation stage.
assert len(squiggle_prepared) == input_row_count, (
    "Datetime preparation changed the row count."
)

assert utc_mismatch_count == 0, (
    "The source time representations do not agree."
)

assert squiggle_prepared["season_year"].eq(
    squiggle_prepared["match_date"].dt.year
).all(), "Some match dates do not agree with season_year."

assert squiggle_prepared.loc[
    completed_match_mask,
    ["home_score", "away_score"],
].notna().all().all(), (
    "Some completed matches are missing final scores."
)

assert squiggle_prepared.loc[
    ~completed_match_mask,
    ["home_score", "away_score"],
].isna().all().all(), (
    "Some non-completed matches contain prepared final scores."
)

squiggle_matches_working = squiggle_prepared

completion_summary = (
    squiggle_matches_working["completion_status"]
    .value_counts()
    .reindex(
        ["completed", "not_started", "in_progress"],
        fill_value=0,
    )
    .rename_axis("completion_status")
    .reset_index(name="records")
)

display(
    squiggle_matches_working[
        [
            "source_game_id",
            "source_datetime",
            "source_utc_offset",
            "source_local_datetime",
            "match_datetime_utc",
            "match_date",
            "start_time",
        ]
    ].head(3)
)

display(completion_summary)

print(f"UTC time mismatches: {utc_mismatch_count}")
print(
    "Local match-date range: "
    f"{squiggle_matches_working['match_date'].min().date()} "
    "to "
    f"{squiggle_matches_working['match_date'].max().date()}"
)
print(
    "Non-completed source score records reset to missing: "
    f"{int((~completed_match_mask).sum())}"
)
print("Squiggle datetime and completion-status check: PASSED")

,source_game_id,source_datetime,source_utc_offset,source_local_datetime,match_datetime_utc,match_date,start_time
0,38494,2026-03-05 19:30:00,+11:00,2026-03-05 19:30:00,2026-03-05 08:30:00+00:00,2026-03-05,19:30:00
1,38495,2026-03-06 20:05:00,+11:00,2026-03-06 19:05:00,2026-03-06 09:05:00+00:00,2026-03-06,19:05:00
2,38496,2026-03-07 16:15:00,+11:00,2026-03-07 16:15:00,2026-03-07 05:15:00+00:00,2026-03-07,16:15:00


,completion_status,records
0,completed,198
1,not_started,20
2,in_progress,0


UTC time mismatches: 0
Local match-date range: 2026-03-05 to 2026-09-26
Non-completed source score records reset to missing: 20
Squiggle datetime and completion-status check: PASSED


**Result**

All Squiggle time values were successfully standardised. The API datetime-plus-offset representation agreed with the Unix timestamp for all 218 records.

Venue-local match dates range from 5 March to 26 September 2026. The snapshot contains 198 completed matches, 20 matches that had not started, and no matches in progress.

The source snapshot represents non-completed matches with zero scores. These source values remain available in `source_home_score` and `source_away_score`, while the prepared `home_score` and `away_score` fields are missing until a match is completed.

The completion status does not yet distinguish assigned future fixtures from unresolved finals placeholders; that classification is performed in Section 4.4.

### 4.4 Identify Completed Matches and Future Fixtures

Completion status alone does not indicate whether a non-completed record is ready for prediction. A future fixture requires assigned home and away teams, while some finals records are placeholders whose participants were unknown on the snapshot date.

Each snapshot record is classified as:

- `completed_match`: the match is complete and both teams are assigned;
- `future_fixture`: the match has not started and both teams are assigned;
- `finals_placeholder`: the match has not started, both teams are unknown, and the record belongs to the finals series;
- `in_progress_match`: the match is currently in progress and both teams are assigned.

Only `future_fixture` records are immediately suitable for future attendance prediction. Finals placeholders remain in the snapshot for traceability but cannot be scored until their participating teams are known.

No records are removed during classification.

In [19]:
# Classify each snapshot record using completion status and team availability.
squiggle_classified = squiggle_matches_working.copy()

teams_assigned_mask = (
    squiggle_classified["home_team_id"].notna()
    & squiggle_classified["away_team_id"].notna()
)

teams_unassigned_mask = (
    squiggle_classified["home_team_id"].isna()
    & squiggle_classified["away_team_id"].isna()
)

completed_mask = (
    squiggle_classified["completion_status"].eq("completed")
)

not_started_mask = (
    squiggle_classified["completion_status"].eq("not_started")
)

in_progress_mask = (
    squiggle_classified["completion_status"].eq("in_progress")
)

squiggle_classified["is_finals_match"] = (
    squiggle_classified["finals_type_code"]
    .ne(0)
    .astype("boolean")
)

record_type = pd.Series(
    pd.NA,
    index=squiggle_classified.index,
    dtype="string",
)

record_type.loc[
    completed_mask & teams_assigned_mask
] = "completed_match"

record_type.loc[
    not_started_mask & teams_assigned_mask
] = "future_fixture"

record_type.loc[
    (
        not_started_mask
        & teams_unassigned_mask
        & squiggle_classified["is_finals_match"]
    )
] = "finals_placeholder"

record_type.loc[
    in_progress_mask & teams_assigned_mask
] = "in_progress_match"

squiggle_classified["snapshot_record_type"] = record_type

# Retain only the checks required to confirm complete classification
# and chronological consistency with the snapshot date.
unclassified_record_count = int(
    squiggle_classified["snapshot_record_type"].isna().sum()
)

completed_after_snapshot_count = int(
    (
        completed_mask
        & squiggle_classified["match_date"].gt(
            squiggle_classified["snapshot_date"]
        )
    ).sum()
)

not_started_before_snapshot_count = int(
    (
        not_started_mask
        & squiggle_classified["match_date"].lt(
            squiggle_classified["snapshot_date"]
        )
    ).sum()
)

assert unclassified_record_count == 0, (
    "Some Squiggle records could not be classified."
)

assert completed_after_snapshot_count == 0, (
    "Some completed matches occur after the snapshot date."
)

assert not_started_before_snapshot_count == 0, (
    "Some non-started matches occur before the snapshot date."
)

squiggle_matches_working = squiggle_classified

# Create convenient views without removing records from the full snapshot.
squiggle_completed_matches = (
    squiggle_matches_working.loc[
        squiggle_matches_working[
            "snapshot_record_type"
        ].eq("completed_match")
    ].copy()
)

squiggle_future_fixtures = (
    squiggle_matches_working.loc[
        squiggle_matches_working[
            "snapshot_record_type"
        ].eq("future_fixture")
    ]
    .sort_values("match_datetime_utc")
    .reset_index(drop=True)
    .copy()
)

squiggle_finals_placeholders = (
    squiggle_matches_working.loc[
        squiggle_matches_working[
            "snapshot_record_type"
        ].eq("finals_placeholder")
    ]
    .sort_values("match_datetime_utc")
    .reset_index(drop=True)
    .copy()
)

record_type_order = [
    "completed_match",
    "future_fixture",
    "finals_placeholder",
    "in_progress_match",
]

record_type_summary = (
    squiggle_matches_working["snapshot_record_type"]
    .value_counts()
    .reindex(record_type_order, fill_value=0)
    .rename_axis("snapshot_record_type")
    .reset_index(name="records")
)

display(record_type_summary)

display(
    squiggle_future_fixtures[
        [
            "source_game_id",
            "round_label",
            "match_date",
            "start_time",
            "home_team",
            "away_team",
            "venue_name",
        ]
    ]
)

print(f"Unclassified records: {unclassified_record_count}")
print(
    "Chronology mismatches: "
    f"{completed_after_snapshot_count + not_started_before_snapshot_count}"
)
print("Squiggle snapshot record classification check: PASSED")

,snapshot_record_type,records
0,completed_match,198
1,future_fixture,9
2,finals_placeholder,11
3,in_progress_match,0


,source_game_id,round_label,match_date,start_time,home_team,away_team,venue_name
0,38697,Round 24,2026-08-20,19:30:00,St Kilda,Gold Coast,Docklands
1,38692,Round 24,2026-08-21,19:40:00,Collingwood,Brisbane Lions,MCG
2,38693,Round 24,2026-08-22,13:15:00,Carlton,Fremantle,Docklands
3,38696,Round 24,2026-08-22,16:15:00,Melbourne,Western Bulldogs,MCG
4,38695,Round 24,2026-08-22,19:45:00,Geelong,Richmond,Kardinia Park
5,38699,Round 24,2026-08-22,19:40:00,Adelaide,Greater Western Sydney,Adelaide Oval
6,38694,Round 24,2026-08-23,12:20:00,Essendon,Port Adelaide,Docklands
7,38698,Round 24,2026-08-23,15:20:00,Sydney,North Melbourne,SCG
8,38700,Round 24,2026-08-23,17:20:00,West Coast,Hawthorn,Perth Stadium


Unclassified records: 0
Chronology mismatches: 0
Squiggle snapshot record classification check: PASSED


**Result**

All 218 Squiggle snapshot records were assigned a clear record type without removing any rows.

The snapshot contains 198 completed matches and nine assigned Round 24 fixtures that are immediately suitable for future attendance prediction. Eleven finals records remain as placeholders because their participating teams were not known on the snapshot date.

No matches were in progress, no records remained unclassified, and no chronological inconsistencies were found relative to the snapshot date.

The full snapshot remains available in `squiggle_matches_working`, while `squiggle_future_fixtures` contains the nine prediction-ready fixtures and `squiggle_finals_placeholders` retains the unresolved finals records.

### 4.5 Create and Validate the Match Identifier

The Squiggle `source_game_id` is unique within the fixed API snapshot, but it is specific to the source system. Assigned 2026 matches therefore receive the same readable project identifier used for historical matches.

The `match_id` format is:

`YYYYMMDD_H##_A##`

It combines the venue-local match date, canonical home-team ID, and canonical away-team ID.

Completed matches and assigned future fixtures can receive this identifier. Finals placeholders cannot receive a valid `match_id` because their participating teams were unknown on the snapshot date. Their project identifier remains missing until the fixture is resolved, while `source_game_id` continues to identify the API record.

In [20]:
# Create the same project match identifier used by the historical table.
squiggle_matches_prepared = squiggle_matches_working.copy()

assigned_match_mask = (
    squiggle_matches_prepared["home_team_id"].notna()
    & squiggle_matches_prepared["away_team_id"].notna()
)

placeholder_match_mask = (
    squiggle_matches_prepared[
        "snapshot_record_type"
    ].eq("finals_placeholder")
)

squiggle_matches_prepared.insert(
    0,
    "match_id",
    pd.Series(
        pd.NA,
        index=squiggle_matches_prepared.index,
        dtype="string",
    ),
)

squiggle_matches_prepared.loc[
    assigned_match_mask,
    "match_id",
] = (
    squiggle_matches_prepared.loc[
        assigned_match_mask,
        "match_date",
    ].dt.strftime("%Y%m%d")
    + "_H"
    + squiggle_matches_prepared.loc[
        assigned_match_mask,
        "home_team_id",
    ]
    .astype("int64")
    .astype("string")
    .str.zfill(2)
    + "_A"
    + squiggle_matches_prepared.loc[
        assigned_match_mask,
        "away_team_id",
    ]
    .astype("int64")
    .astype("string")
    .str.zfill(2)
)

# Validate source and project identifiers without assigning artificial
# match IDs to unresolved finals placeholders.
duplicate_match_id_rows = int(
    squiggle_matches_prepared.loc[
        assigned_match_mask,
        "match_id",
    ]
    .duplicated(keep=False)
    .sum()
)

assert (
    squiggle_matches_prepared["source_game_id"].notna().all()
    and squiggle_matches_prepared["source_game_id"].is_unique
), "source_game_id is missing or duplicated within the snapshot."

assert duplicate_match_id_rows == 0, (
    "The assigned Squiggle match_id values are not unique."
)

assert squiggle_matches_prepared.loc[
    assigned_match_mask,
    "match_id",
].notna().all(), (
    "Some assigned matches do not have a match_id."
)

assert squiggle_matches_prepared.loc[
    placeholder_match_mask,
    "match_id",
].isna().all(), (
    "A finals placeholder was assigned an artificial match_id."
)

# Refresh the classified views so that assigned records include match_id.
squiggle_completed_matches = (
    squiggle_matches_prepared.loc[
        squiggle_matches_prepared[
            "snapshot_record_type"
        ].eq("completed_match")
    ].copy()
)

squiggle_future_fixtures = (
    squiggle_matches_prepared.loc[
        squiggle_matches_prepared[
            "snapshot_record_type"
        ].eq("future_fixture")
    ]
    .sort_values("match_datetime_utc")
    .reset_index(drop=True)
    .copy()
)

squiggle_finals_placeholders = (
    squiggle_matches_prepared.loc[
        placeholder_match_mask
    ]
    .sort_values("match_datetime_utc")
    .reset_index(drop=True)
    .copy()
)

identifier_summary = pd.DataFrame(
    {
        "check": [
            "Snapshot records",
            "Unique source_game_id values",
            "Assigned records with match_id",
            "Duplicate assigned match_id rows",
            "Placeholders without match_id",
        ],
        "value": [
            len(squiggle_matches_prepared),
            squiggle_matches_prepared[
                "source_game_id"
            ].nunique(),
            int(
                squiggle_matches_prepared.loc[
                    assigned_match_mask,
                    "match_id",
                ].notna().sum()
            ),
            duplicate_match_id_rows,
            int(
                squiggle_matches_prepared.loc[
                    placeholder_match_mask,
                    "match_id",
                ].isna().sum()
            ),
        ],
        "status": [
            "PASSED",
            "PASSED",
            "PASSED",
            "PASSED",
            "EXPECTED",
        ],
    }
)

display(
    squiggle_future_fixtures[
        [
            "match_id",
            "source_game_id",
            "match_date",
            "home_team",
            "away_team",
        ]
    ].head(3)
)

display(identifier_summary)

print("Squiggle match identifier check: PASSED")

,match_id,source_game_id,match_date,home_team,away_team
0,20260820_H15_A08,38697,2026-08-20,St Kilda,Gold Coast
1,20260821_H04_A02,38692,2026-08-21,Collingwood,Brisbane Lions
2,20260822_H03_A06,38693,2026-08-22,Carlton,Fremantle


,check,value,status
0,Snapshot records,218,PASSED
1,Unique source_game_id values,218,PASSED
2,Assigned records with match_id,207,PASSED
3,Duplicate assigned match_id rows,0,PASSED
4,Placeholders without match_id,11,EXPECTED


Squiggle match identifier check: PASSED


**Result**

All 218 Squiggle records have unique source identifiers within the fixed API snapshot.

A unique project `match_id` was created for all 207 records with assigned teams, using the same date–home-team–away-team format as the historical match table. No duplicate assigned match identifiers were found.

The 11 unresolved finals placeholders retain their Squiggle `source_game_id` values but have missing `match_id` values because their participating teams were unknown on the snapshot date.

The completed Section 4 output is stored in `squiggle_matches_prepared`.

## 5. Prepare the School-Holiday Table

### 5.1 Prepare the 2004-2023 Daily Holiday Records

The historical dataset contains one record for each capital city and calendar date from 2004 to 2023.

The documented `schoolhols` field is retained as the binary school-holiday indicator required by the v1 model. The `school.hols` field identifies the numbered holiday period but is not required for the current prediction feature, so it is excluded rather than unnecessarily imputing its single missing classification.

City names, state or territory codes, dates, and data types are standardised before the historical records are combined with the official 2024–2026 calendars.

In [21]:
city_to_state_code = {
    "Adelaide": "SA",
    "Brisbane": "QLD",
    "Canberra": "ACT",
    "Darwin": "NT",
    "Hobart": "TAS",
    "Melbourne": "VIC",
    "Perth": "WA",
    "Sydney": "NSW",
}

# Retain only the fields required for the v1 school-holiday feature.
historical_school_holidays = (
    historical_school_holidays_raw.loc[:, ["City", "Date", "schoolhols"]]
    .rename(
        columns={
            "City": "city",
            "Date": "calendar_date",
            "schoolhols": "is_school_holiday",
        }
    )
    .copy()
)

# Standardise city labels and assign the corresponding jurisdiction.
historical_school_holidays["city"] = (
    historical_school_holidays["city"]
    .astype("string")
    .str.strip()
)

historical_school_holidays["state_code"] = (
    historical_school_holidays["city"]
    .map(city_to_state_code)
    .astype("string")
)

# Convert dates and the documented binary holiday flag.
historical_school_holidays["calendar_date"] = pd.to_datetime(
    historical_school_holidays["calendar_date"],
    errors="raise",
)

historical_school_holidays["is_school_holiday"] = pd.to_numeric(
    historical_school_holidays["is_school_holiday"].astype("string"),
    errors="raise",
).astype("Int8")

# Retain a simple source label for the later multi-source combination.
historical_school_holidays["source_name"] = "uoa_figshare"
historical_school_holidays["source_name"] = (
    historical_school_holidays["source_name"].astype("string")
)

historical_school_holidays = (
    historical_school_holidays[
        [
            "city",
            "state_code",
            "calendar_date",
            "is_school_holiday",
            "source_name",
        ]
    ]
    .sort_values(["calendar_date", "city"])
    .reset_index(drop=True)
)

display(historical_school_holidays.head(8))

,city,state_code,calendar_date,is_school_holiday,source_name
0,Adelaide,SA,2004-01-01,1,uoa_figshare
1,Brisbane,QLD,2004-01-01,1,uoa_figshare
2,Canberra,ACT,2004-01-01,1,uoa_figshare
3,Darwin,NT,2004-01-01,1,uoa_figshare
4,Hobart,TAS,2004-01-01,1,uoa_figshare
5,Melbourne,VIC,2004-01-01,1,uoa_figshare
6,Perth,WA,2004-01-01,1,uoa_figshare
7,Sydney,NSW,2004-01-01,1,uoa_figshare


The prepared historical table is validated at its intended grain of one city and one calendar date per row. The checks confirm complete daily coverage, valid jurisdiction mappings, unique city-date records, and binary holiday values.

In [22]:
expected_dates = pd.date_range(
    start="2004-01-01",
    end="2023-12-31",
    freq="D",
)

city_day_counts = (
    historical_school_holidays
    .groupby("city")["calendar_date"]
    .nunique()
)

actual_dates = pd.DatetimeIndex(
    historical_school_holidays["calendar_date"].unique()
).sort_values()

complete_daily_panel = (
    actual_dates.equals(expected_dates)
    and city_day_counts.eq(len(expected_dates)).all()
)

critical_columns = [
    "city",
    "state_code",
    "calendar_date",
    "is_school_holiday",
]

validation_5_1 = pd.DataFrame(
    {
        "check": [
            "Expected row count",
            "Capital cities present",
            "Complete daily coverage",
            "Duplicate city-date rows",
            "Unmapped city values",
            "Unexpected holiday values",
            "Missing critical values",
        ],
        "value": [
            len(historical_school_holidays),
            historical_school_holidays["city"].nunique(),
            complete_daily_panel,
            historical_school_holidays.duplicated(
                ["city", "calendar_date"]
            ).sum(),
            historical_school_holidays["state_code"].isna().sum(),
            (~historical_school_holidays["is_school_holiday"].isin([0, 1])).sum(),
            historical_school_holidays[critical_columns].isna().sum().sum(),
        ],
        "expected": [
            len(expected_dates) * 8,
            8,
            True,
            0,
            0,
            0,
            0,
        ],
    }
)

validation_5_1["status"] = (
    validation_5_1["value"]
    .eq(validation_5_1["expected"])
    .map({True: "PASSED", False: "FAILED"})
)

display(validation_5_1)

if not validation_5_1["status"].eq("PASSED").all():
    raise ValueError(
        "Historical school-holiday preparation check failed."
    )

print("Historical school-holiday preparation check: PASSED")

,check,value,expected,status
0,Expected row count,58440,58440,PASSED
1,Capital cities present,8,8,PASSED
2,Complete daily coverage,True,True,PASSED
3,Duplicate city-date rows,0,0,PASSED
4,Unmapped city values,0,0,PASSED
5,Unexpected holiday values,0,0,PASSED
6,Missing critical values,0,0,PASSED


Historical school-holiday preparation check: PASSED


#### Result

The historical school-holiday table contains 58,440 unique city-date records covering eight Australian capital cities from 1 January 2004 to 31 December 2023.

All cities were mapped to their corresponding state or territory, daily coverage is complete, and the binary holiday indicator contains no missing or unexpected values.

The optional `school.hols` period classification was excluded because the v1 model requires only the documented binary school-holiday indicator.

### 5.2 Structure the Reviewed 2024–2026 Calendar Periods

The reviewed 2024–2026 school-calendar dates are recorded in a compact structured register. Each record represents one jurisdiction and calendar year, with four term start and end dates.

The following scope decisions are applied:

- Sydney uses the NSW Eastern Division calendar.
- Darwin uses the standard Northern Territory government-school dates; the separate Gunbalanya calendar is excluded.
- Each jurisdiction-year record contains four school-term periods.
- The dates were checked against the corresponding source calendar before entry.
- The original source documents remain unchanged under `data/raw/`.

In [23]:
# Record the reviewed term boundaries for each jurisdiction and calendar year.

reviewed_term_rows = [
    ("ACT", 2024,
     "2024-01-29", "2024-04-12", "2024-04-29", "2024-07-05",
     "2024-07-22", "2024-09-27", "2024-10-14", "2024-12-17"),
    ("ACT", 2025,
     "2025-01-31", "2025-04-11", "2025-04-28", "2025-07-04",
     "2025-07-21", "2025-09-26", "2025-10-13", "2025-12-18"),
    ("ACT", 2026,
     "2026-01-29", "2026-04-02", "2026-04-20", "2026-07-03",
     "2026-07-20", "2026-09-25", "2026-10-12", "2026-12-18"),

    ("NSW", 2024,
     "2024-01-30", "2024-04-12", "2024-04-29", "2024-07-05",
     "2024-07-22", "2024-09-27", "2024-10-14", "2024-12-20"),
    ("NSW", 2025,
     "2025-01-31", "2025-04-11", "2025-04-28", "2025-07-04",
     "2025-07-21", "2025-09-26", "2025-10-13", "2025-12-19"),
    ("NSW", 2026,
     "2026-01-27", "2026-04-02", "2026-04-20", "2026-07-03",
     "2026-07-20", "2026-09-25", "2026-10-12", "2026-12-17"),

    ("NT", 2024,
     "2024-01-30", "2024-04-05", "2024-04-16", "2024-06-21",
     "2024-07-16", "2024-09-20", "2024-10-08", "2024-12-12"),
    ("NT", 2025,
     "2025-01-29", "2025-04-04", "2025-04-15", "2025-06-20",
     "2025-07-15", "2025-09-19", "2025-10-07", "2025-12-11"),
    ("NT", 2026,
     "2026-01-29", "2026-04-02", "2026-04-14", "2026-06-19",
     "2026-07-14", "2026-09-18", "2026-10-06", "2026-12-10"),

    ("QLD", 2024,
     "2024-01-22", "2024-03-28", "2024-04-15", "2024-06-21",
     "2024-07-08", "2024-09-13", "2024-09-30", "2024-12-13"),
    ("QLD", 2025,
     "2025-01-28", "2025-04-04", "2025-04-22", "2025-06-27",
     "2025-07-14", "2025-09-19", "2025-10-07", "2025-12-12"),
    ("QLD", 2026,
     "2026-01-27", "2026-04-02", "2026-04-20", "2026-06-26",
     "2026-07-13", "2026-09-18", "2026-10-06", "2026-12-11"),

    ("SA", 2024,
     "2024-01-29", "2024-04-12", "2024-04-29", "2024-07-05",
     "2024-07-22", "2024-09-27", "2024-10-14", "2024-12-13"),
    ("SA", 2025,
     "2025-01-28", "2025-04-11", "2025-04-28", "2025-07-04",
     "2025-07-21", "2025-09-26", "2025-10-13", "2025-12-12"),
    ("SA", 2026,
     "2026-01-27", "2026-04-10", "2026-04-27", "2026-07-03",
     "2026-07-20", "2026-09-25", "2026-10-12", "2026-12-11"),

    ("TAS", 2024,
     "2024-02-08", "2024-04-12", "2024-04-29", "2024-07-05",
     "2024-07-22", "2024-09-27", "2024-10-14", "2024-12-19"),
    ("TAS", 2025,
     "2025-02-06", "2025-04-11", "2025-04-28", "2025-07-04",
     "2025-07-21", "2025-09-26", "2025-10-13", "2025-12-18"),
    ("TAS", 2026,
     "2026-02-05", "2026-04-17", "2026-05-04", "2026-07-10",
     "2026-07-27", "2026-10-02", "2026-10-19", "2026-12-18"),

    ("VIC", 2024,
     "2024-01-29", "2024-03-28", "2024-04-15", "2024-06-28",
     "2024-07-15", "2024-09-20", "2024-10-07", "2024-12-20"),
    ("VIC", 2025,
     "2025-01-28", "2025-04-04", "2025-04-22", "2025-07-04",
     "2025-07-21", "2025-09-19", "2025-10-06", "2025-12-19"),
    ("VIC", 2026,
     "2026-01-27", "2026-04-02", "2026-04-20", "2026-06-26",
     "2026-07-13", "2026-09-18", "2026-10-05", "2026-12-18"),

    ("WA", 2024,
     "2024-01-31", "2024-03-28", "2024-04-15", "2024-06-28",
     "2024-07-15", "2024-09-20", "2024-10-07", "2024-12-12"),
    ("WA", 2025,
     "2025-02-05", "2025-04-11", "2025-04-28", "2025-07-04",
     "2025-07-21", "2025-09-26", "2025-10-13", "2025-12-18"),
    ("WA", 2026,
     "2026-02-02", "2026-04-02", "2026-04-20", "2026-07-03",
     "2026-07-20", "2026-09-25", "2026-10-12", "2026-12-17"),
]

reviewed_term_columns = [
    "state_code",
    "calendar_year",
    "term_1_start",
    "term_1_end",
    "term_2_start",
    "term_2_end",
    "term_3_start",
    "term_3_end",
    "term_4_start",
    "term_4_end",
]

reviewed_term_calendar_wide = pd.DataFrame(
    reviewed_term_rows,
    columns=reviewed_term_columns,
)

print(
    "Reviewed jurisdiction-year records: "
    f"{len(reviewed_term_calendar_wide):,}"
)

display(reviewed_term_calendar_wide.head(8))

Reviewed jurisdiction-year records: 24


,state_code,calendar_year,term_1_start,term_1_end,term_2_start,term_2_end,term_3_start,term_3_end,term_4_start,term_4_end
0,ACT,2024,2024-01-29,2024-04-12,2024-04-29,2024-07-05,2024-07-22,2024-09-27,2024-10-14,2024-12-17
1,ACT,2025,2025-01-31,2025-04-11,2025-04-28,2025-07-04,2025-07-21,2025-09-26,2025-10-13,2025-12-18
2,ACT,2026,2026-01-29,2026-04-02,2026-04-20,2026-07-03,2026-07-20,2026-09-25,2026-10-12,2026-12-18
3,NSW,2024,2024-01-30,2024-04-12,2024-04-29,2024-07-05,2024-07-22,2024-09-27,2024-10-14,2024-12-20
4,NSW,2025,2025-01-31,2025-04-11,2025-04-28,2025-07-04,2025-07-21,2025-09-26,2025-10-13,2025-12-19
5,NSW,2026,2026-01-27,2026-04-02,2026-04-20,2026-07-03,2026-07-20,2026-09-25,2026-10-12,2026-12-17
6,NT,2024,2024-01-30,2024-04-05,2024-04-16,2024-06-21,2024-07-16,2024-09-20,2024-10-08,2024-12-12
7,NT,2025,2025-01-29,2025-04-04,2025-04-15,2025-06-20,2025-07-15,2025-09-19,2025-10-07,2025-12-11


The jurisdiction-year register is reshaped into a long-format table so that each row represents one school-term period.

This structure makes the term dates easier to validate and later expand into daily calendar records.

In [24]:
# Reshape the four term pairs into one row per school-term period.

term_period_frames = []

for term_number in range(1, 5):
    term_period = reviewed_term_calendar_wide[
        [
            "state_code",
            "calendar_year",
            f"term_{term_number}_start",
            f"term_{term_number}_end",
        ]
    ].copy()

    term_period = term_period.rename(
        columns={
            f"term_{term_number}_start": "term_start_date",
            f"term_{term_number}_end": "term_end_date",
        }
    )

    term_period["term_number"] = term_number
    term_period_frames.append(term_period)

reviewed_school_term_periods = pd.concat(
    term_period_frames,
    ignore_index=True,
)

# Associate each jurisdiction with the capital city used in the project.

state_to_city = {
    "ACT": "Canberra",
    "NSW": "Sydney",
    "NT": "Darwin",
    "QLD": "Brisbane",
    "SA": "Adelaide",
    "TAS": "Hobart",
    "VIC": "Melbourne",
    "WA": "Perth",
}

reviewed_school_term_periods["city"] = (
    reviewed_school_term_periods["state_code"].map(state_to_city)
)

# Convert the reviewed date values to datetime columns.

date_columns = ["term_start_date", "term_end_date"]

for column in date_columns:
    reviewed_school_term_periods[column] = pd.to_datetime(
        reviewed_school_term_periods[column],
        errors="raise",
    )

# Add a concise source label for the prepared reference table.

reviewed_school_term_periods["source_name"] = (
    "Reviewed state and territory school calendars"
)

# Arrange the fields and records into a consistent structure.

reviewed_school_term_periods = (
    reviewed_school_term_periods[
        [
            "state_code",
            "city",
            "calendar_year",
            "term_number",
            "term_start_date",
            "term_end_date",
            "source_name",
        ]
    ]
    .sort_values(
        ["state_code", "calendar_year", "term_number"]
    )
    .reset_index(drop=True)
)

print(
    "Reviewed term-period records: "
    f"{len(reviewed_school_term_periods):,}"
)

display(reviewed_school_term_periods.head(12))

Reviewed term-period records: 96


,state_code,city,calendar_year,term_number,term_start_date,term_end_date,source_name
0,ACT,Canberra,2024,1,2024-01-29,2024-04-12,Reviewed state and territory school calendars
1,ACT,Canberra,2024,2,2024-04-29,2024-07-05,Reviewed state and territory school calendars
2,ACT,Canberra,2024,3,2024-07-22,2024-09-27,Reviewed state and territory school calendars
3,ACT,Canberra,2024,4,2024-10-14,2024-12-17,Reviewed state and territory school calendars
4,ACT,Canberra,2025,1,2025-01-31,2025-04-11,Reviewed state and territory school calendars
5,ACT,Canberra,2025,2,2025-04-28,2025-07-04,Reviewed state and territory school calendars
6,ACT,Canberra,2025,3,2025-07-21,2025-09-26,Reviewed state and territory school calendars
7,ACT,Canberra,2025,4,2025-10-13,2025-12-18,Reviewed state and territory school calendars
8,ACT,Canberra,2026,1,2026-01-29,2026-04-02,Reviewed state and territory school calendars
9,ACT,Canberra,2026,2,2026-04-20,2026-07-03,Reviewed state and territory school calendars


The prepared term-period table is validated before it is used to create daily calendar records.

The checks confirm that every jurisdiction-year contains four terms, the term keys are unique, and all date ranges are complete, correctly ordered, and non-overlapping.

In [25]:
# Prepare the term periods in chronological order for validation.

term_periods_for_validation = (
    reviewed_school_term_periods
    .sort_values(
        ["state_code", "calendar_year", "term_number"]
    )
    .copy()
)

# Count the number of terms recorded for each jurisdiction-year.

term_counts = (
    term_periods_for_validation
    .groupby(["state_code", "calendar_year"])
    .size()
)

jurisdiction_year_groups = len(term_counts)
groups_with_four_terms = int(term_counts.eq(4).sum())

# Check whether any term starts before the preceding term has ended.

previous_term_end = (
    term_periods_for_validation
    .groupby(["state_code", "calendar_year"])["term_end_date"]
    .shift()
)

overlapping_period_count = int(
    (
        previous_term_end.notna()
        & term_periods_for_validation["term_start_date"].le(
            previous_term_end
        )
    ).sum()
)

# Calculate the remaining key and date-quality measures.

duplicate_term_key_count = int(
    term_periods_for_validation.duplicated(
        subset=["state_code", "calendar_year", "term_number"]
    ).sum()
)

invalid_date_range_count = int(
    (
        term_periods_for_validation["term_start_date"]
        > term_periods_for_validation["term_end_date"]
    ).sum()
)

year_mismatch_count = int(
    (
        term_periods_for_validation["term_start_date"].dt.year.ne(
            term_periods_for_validation["calendar_year"]
        )
        | term_periods_for_validation["term_end_date"].dt.year.ne(
            term_periods_for_validation["calendar_year"]
        )
    ).sum()
)

critical_columns = [
    "state_code",
    "city",
    "calendar_year",
    "term_number",
    "term_start_date",
    "term_end_date",
    "source_name",
]

missing_critical_value_count = int(
    term_periods_for_validation[critical_columns]
    .isna()
    .sum()
    .sum()
)

# Summarise the validation results in a portfolio-friendly table.

term_period_validation = pd.DataFrame(
    {
        "check": [
            "Expected term-period rows",
            "Jurisdiction-year groups",
            "Groups with four terms",
            "Capital cities present",
            "Duplicate term keys",
            "Invalid term date ranges",
            "Overlapping term periods",
            "Calendar-year mismatches",
            "Missing critical values",
        ],
        "value": [
            len(term_periods_for_validation),
            jurisdiction_year_groups,
            groups_with_four_terms,
            term_periods_for_validation["city"].nunique(),
            duplicate_term_key_count,
            invalid_date_range_count,
            overlapping_period_count,
            year_mismatch_count,
            missing_critical_value_count,
        ],
        "expected": [
            96,
            24,
            24,
            8,
            0,
            0,
            0,
            0,
            0,
        ],
    }
)

term_period_validation["status"] = (
    term_period_validation["value"]
    .eq(term_period_validation["expected"])
    .map({True: "PASSED", False: "FAILED"})
)

display(term_period_validation)

if term_period_validation["status"].ne("PASSED").any():
    raise ValueError(
        "The reviewed school-term period table failed validation."
    )

print("Reviewed school-term period check: PASSED")

,check,value,expected,status
0,Expected term-period rows,96,96,PASSED
1,Jurisdiction-year groups,24,24,PASSED
2,Groups with four terms,24,24,PASSED
3,Capital cities present,8,8,PASSED
4,Duplicate term keys,0,0,PASSED
5,Invalid term date ranges,0,0,PASSED
6,Overlapping term periods,0,0,PASSED
7,Calendar-year mismatches,0,0,PASSED
8,Missing critical values,0,0,PASSED


Reviewed school-term period check: PASSED


### 5.3 Expand Official Holiday Periods to Daily Records

The reviewed term periods are expanded into a complete daily panel for the eight capital cities from 2024 to 2026.

A date is marked as a school holiday when it falls outside all four term periods for the corresponding city and calendar year. Term start and end dates are treated as inclusive.

The resulting indicator represents school-vacation status rather than general weekend or public-holiday status.

In [26]:
# Create a complete daily date range covering 2024 to 2026.

reviewed_calendar_dates = pd.date_range(
    start="2024-01-01",
    end="2026-12-31",
    freq="D",
)

# Build one complete daily calendar for each capital city.

daily_calendar_frames = []

for state_code, city in state_to_city.items():
    city_calendar = pd.DataFrame(
        {
            "city": city,
            "state_code": state_code,
            "calendar_date": reviewed_calendar_dates,
        }
    )

    daily_calendar_frames.append(city_calendar)

reviewed_school_holidays_daily = pd.concat(
    daily_calendar_frames,
    ignore_index=True,
)

# Initially classify every date as outside the school-term periods.

reviewed_school_holidays_daily["is_school_holiday"] = True

# Reclassify dates within each reviewed term period as non-holiday dates.

for term_period in reviewed_school_term_periods.itertuples(index=False):
    term_date_mask = (
        reviewed_school_holidays_daily["state_code"].eq(
            term_period.state_code
        )
        & reviewed_school_holidays_daily["calendar_date"].between(
            term_period.term_start_date,
            term_period.term_end_date,
            inclusive="both",
        )
    )

    reviewed_school_holidays_daily.loc[
        term_date_mask,
        "is_school_holiday",
    ] = False

# Add the source label used for the reviewed calendar records.

reviewed_school_holidays_daily["source_name"] = (
    "Reviewed state and territory school calendars"
)

# Arrange the fields to match the historical daily table.

reviewed_school_holidays_daily = (
    reviewed_school_holidays_daily[
        [
            "city",
            "state_code",
            "calendar_date",
            "is_school_holiday",
            "source_name",
        ]
    ]
    .sort_values(["calendar_date", "city"])
    .reset_index(drop=True)
)

print(
    "Reviewed daily school-holiday records: "
    f"{len(reviewed_school_holidays_daily):,}"
)

print(
    "Date range: "
    f"{reviewed_school_holidays_daily['calendar_date'].min().date()} "
    "to "
    f"{reviewed_school_holidays_daily['calendar_date'].max().date()}"
)

print(
    "Capital cities: "
    f"{reviewed_school_holidays_daily['city'].nunique()}"
)

display(reviewed_school_holidays_daily.head(8))

Reviewed daily school-holiday records: 8,768
Date range: 2024-01-01 to 2026-12-31
Capital cities: 8


,city,state_code,calendar_date,is_school_holiday,source_name
0,Adelaide,SA,2024-01-01,True,Reviewed state and territory school calendars
1,Brisbane,QLD,2024-01-01,True,Reviewed state and territory school calendars
2,Canberra,ACT,2024-01-01,True,Reviewed state and territory school calendars
3,Darwin,NT,2024-01-01,True,Reviewed state and territory school calendars
4,Hobart,TAS,2024-01-01,True,Reviewed state and territory school calendars
5,Melbourne,VIC,2024-01-01,True,Reviewed state and territory school calendars
6,Perth,WA,2024-01-01,True,Reviewed state and territory school calendars
7,Sydney,NSW,2024-01-01,True,Reviewed state and territory school calendars


### 5.4 Combine and Validate the Holiday Coverage

The prepared historical records for 2004–2023 are combined with the reviewed daily records for 2024–2026.

Both sources use the same city-date grain and column structure. The combined table provides continuous school-holiday coverage for the eight capital cities from 2004 through 2026.

In [27]:
# Select the common fields used by both daily calendar sources.

holiday_daily_columns = [
    "city",
    "state_code",
    "calendar_date",
    "is_school_holiday",
    "source_name",
]

# Combine the historical and reviewed daily school-holiday records.

school_holidays_daily = pd.concat(
    [
        historical_school_holidays[holiday_daily_columns],
        reviewed_school_holidays_daily[holiday_daily_columns],
    ],
    ignore_index=True,
)

# Apply consistent data types across the combined table.

school_holidays_daily["calendar_date"] = pd.to_datetime(
    school_holidays_daily["calendar_date"],
    errors="raise",
)

# Normalise the historical 0/1 values and the reviewed True/False values.

normalised_holiday_values = (
    school_holidays_daily["is_school_holiday"]
    .astype("string")
    .str.strip()
    .str.lower()
)

holiday_value_map = {
    "0": False,
    "1": True,
    "false": False,
    "true": True,
}

if not normalised_holiday_values.dropna().isin(
    holiday_value_map
).all():
    raise ValueError(
        "Unexpected school-holiday values were found."
    )

school_holidays_daily["is_school_holiday"] = (
    normalised_holiday_values
    .map(holiday_value_map)
    .astype("boolean")
)

for column in ["city", "state_code", "source_name"]:
    school_holidays_daily[column] = (
        school_holidays_daily[column].astype("string")
    )

# Arrange the final records by date and city.

school_holidays_daily = (
    school_holidays_daily
    .sort_values(["calendar_date", "city"])
    .reset_index(drop=True)
)

# Summarise the contribution of each source.

holiday_source_summary = (
    school_holidays_daily
    .groupby("source_name", observed=True)
    .agg(
        records=("calendar_date", "size"),
        start_date=("calendar_date", "min"),
        end_date=("calendar_date", "max"),
    )
    .reset_index()
)

print(
    "Combined daily school-holiday records: "
    f"{len(school_holidays_daily):,}"
)

print(
    "Combined date range: "
    f"{school_holidays_daily['calendar_date'].min().date()} "
    "to "
    f"{school_holidays_daily['calendar_date'].max().date()}"
)

display(holiday_source_summary)

Combined daily school-holiday records: 67,208
Combined date range: 2004-01-01 to 2026-12-31


,source_name,records,start_date,end_date
0,Reviewed state and territory school calendars,8768,2024-01-01,2026-12-31
1,uoa_figshare,58440,2004-01-01,2023-12-31


The combined table is checked for complete city-date coverage, unique city-date keys, and missing critical values. These checks confirm that the table can be safely joined to match records without introducing missing or duplicated matches.

In [28]:
# Summarise the date coverage available for each capital city.

expected_start_date = pd.Timestamp("2004-01-01")
expected_end_date = pd.Timestamp("2026-12-31")

expected_dates_per_city = len(
    pd.date_range(
        expected_start_date,
        expected_end_date,
        freq="D",
    )
)

coverage_by_city = (
    school_holidays_daily
    .groupby("city", observed=True)
    .agg(
        unique_dates=("calendar_date", "nunique"),
        start_date=("calendar_date", "min"),
        end_date=("calendar_date", "max"),
    )
)

complete_daily_coverage = bool(
    len(coverage_by_city) == 8
    and coverage_by_city["unique_dates"]
    .eq(expected_dates_per_city)
    .all()
    and coverage_by_city["start_date"]
    .eq(expected_start_date)
    .all()
    and coverage_by_city["end_date"]
    .eq(expected_end_date)
    .all()
)

# Calculate the key validation measures.

duplicate_city_date_count = int(
    school_holidays_daily.duplicated(
        subset=["city", "calendar_date"]
    ).sum()
)

missing_critical_value_count = int(
    school_holidays_daily[
        [
            "city",
            "state_code",
            "calendar_date",
            "is_school_holiday",
        ]
    ]
    .isna()
    .sum()
    .sum()
)

holiday_classes_present = int(
    school_holidays_daily["is_school_holiday"]
    .dropna()
    .nunique()
)

# Present a concise validation summary.

combined_holiday_validation = pd.DataFrame(
    {
        "check": [
            "Expected daily rows",
            "Capital cities present",
            "Complete daily coverage",
            "Duplicate city-date rows",
            "Holiday classes present",
            "Missing critical values",
        ],
        "value": [
            len(school_holidays_daily),
            school_holidays_daily["city"].nunique(),
            complete_daily_coverage,
            duplicate_city_date_count,
            holiday_classes_present,
            missing_critical_value_count,
        ],
        "expected": [
            67_208,
            8,
            True,
            0,
            2,
            0,
        ],
    }
)

combined_holiday_validation["status"] = (
    combined_holiday_validation["value"]
    .eq(combined_holiday_validation["expected"])
    .map({True: "PASSED", False: "FAILED"})
)

display(combined_holiday_validation)

if combined_holiday_validation["status"].ne("PASSED").any():
    raise ValueError(
        "The combined school-holiday table failed validation."
    )

print("Combined school-holiday coverage check: PASSED")

,check,value,expected,status
0,Expected daily rows,67208,67208,PASSED
1,Capital cities present,8,8,PASSED
2,Complete daily coverage,True,True,PASSED
3,Duplicate city-date rows,0,0,PASSED
4,Holiday classes present,2,2,PASSED
5,Missing critical values,0,0,PASSED


Combined school-holiday coverage check: PASSED


All validation checks passed. The combined table contains 67,208 unique city-date records covering eight capital cities from 1 January 2004 through 31 December 2026.

Both school-holiday and non-holiday values are present, and no critical fields are missing. The prepared table is ready for later match-date feature integration.

## 6. Export and Summarise the Results

### 6.1 Export the Database-Ready Tables

Only the validated tables required for subsequent PostgreSQL loading are exported. The source files under `data/raw/` remain unchanged, and intermediate diagnostic tables are not exported.

In [29]:
from pathlib import Path

# Resolve the project root from either the repository root or notebooks folder.

current_directory = Path.cwd().resolve()

project_root = (
    current_directory.parent
    if current_directory.name.lower() == "notebooks"
    else current_directory
)

if not (project_root / "data").exists():
    raise FileNotFoundError(
        "The project data directory could not be located."
    )

# Create the database-ready output directory if required.

export_directory = (
    project_root
    / "data"
    / "interim"
    / "postgres_ready"
)

export_directory.mkdir(
    parents=True,
    exist_ok=True,
)

# Define only the final validated tables required downstream.

export_tables = {
    "historical_matches.csv": historical_matches_prepared,
    "squiggle_matches_2026_snapshot.csv": squiggle_matches_prepared,
    "team_reference.csv": team_reference,
    "venue_reference.csv": venue_reference,
    "school_holidays_daily.csv": school_holidays_daily,
}

# Export each table without the pandas index.

for file_name, table in export_tables.items():
    output_path = export_directory / file_name

    table.to_csv(
        output_path,
        index=False,
        encoding="utf-8",
    )

    print(
        f"Exported {file_name}: "
        f"{len(table):,} rows"
    )

print(f"\nExport directory: {export_directory}")

Exported historical_matches.csv: 2,879 rows
Exported squiggle_matches_2026_snapshot.csv: 218 rows
Exported team_reference.csv: 18 rows
Exported venue_reference.csv: 27 rows
Exported school_holidays_daily.csv: 67,208 rows

Export directory: D:\Projects\AFL_Project\data\interim\postgres_ready


### 6.2 Summarise the Output Tables

The exported tables are summarised by record grain, key, dimensions, and date coverage. This provides a concise handoff reference for the subsequent PostgreSQL loading step.

In [30]:
# Summarise the final database-ready output tables.

output_table_summary = pd.DataFrame(
    [
        {
            "file_name": "historical_matches.csv",
            "record_grain": "One completed historical match",
            "rows": len(historical_matches_prepared),
            "columns": historical_matches_prepared.shape[1],
            "key": "match_id",
            "date_coverage": (
                f"{historical_matches_prepared['match_date'].min().date()} "
                "to "
                f"{historical_matches_prepared['match_date'].max().date()}"
            ),
            "status": "PASSED",
        },
        {
            "file_name": "squiggle_matches_2026_snapshot.csv",
            "record_grain": "One match record per API snapshot",
            "rows": len(squiggle_matches_prepared),
            "columns": squiggle_matches_prepared.shape[1],
            "key": "snapshot_date + source_game_id",
            "date_coverage": (
                f"{squiggle_matches_prepared['match_date'].min().date()} "
                "to "
                f"{squiggle_matches_prepared['match_date'].max().date()}"
            ),
            "status": "PASSED",
        },
        {
            "file_name": "team_reference.csv",
            "record_grain": "One AFL team",
            "rows": len(team_reference),
            "columns": team_reference.shape[1],
            "key": "team_id",
            "date_coverage": "Not applicable",
            "status": "PASSED",
        },
        {
            "file_name": "venue_reference.csv",
            "record_grain": "One standardised venue",
            "rows": len(venue_reference),
            "columns": venue_reference.shape[1],
            "key": "venue_name",
            "date_coverage": "Not applicable",
            "status": "PASSED",
        },
        {
            "file_name": "school_holidays_daily.csv",
            "record_grain": "One capital city-date",
            "rows": len(school_holidays_daily),
            "columns": school_holidays_daily.shape[1],
            "key": "city + calendar_date",
            "date_coverage": (
                f"{school_holidays_daily['calendar_date'].min().date()} "
                "to "
                f"{school_holidays_daily['calendar_date'].max().date()}"
            ),
            "status": "PASSED",
        },
    ]
)

display(output_table_summary)

,file_name,record_grain,rows,columns,key,date_coverage,status
0,historical_matches.csv,One completed historical match,2879,19,match_id,2012-03-24 to 2025-09-27,PASSED
1,squiggle_matches_2026_snapshot.csv,One match record per API snapshot,218,37,snapshot_date + source_game_id,2026-03-05 to 2026-09-26,PASSED
2,team_reference.csv,One AFL team,18,3,team_id,Not applicable,PASSED
3,venue_reference.csv,One standardised venue,27,5,venue_name,Not applicable,PASSED
4,school_holidays_daily.csv,One capital city-date,67208,5,city + calendar_date,2004-01-01 to 2026-12-31,PASSED


### 6.3 Document Limitations and Confirm the Next Step

The five database-ready tables were exported successfully and passed their source-specific validation checks.

The current v1 dataset has the following limitations:

- The match and calendar data are obtained from publicly available sources rather than internal club systems.
- School-holiday coverage is represented by the eight capital-city calendars; regional calendar variations are not modelled.
- The Squiggle table is a point-in-time snapshot captured on 17 August 2026, so later fixture updates are not reflected.
- Eleven future finals records remain placeholders because their participating teams have not yet been determined.
- International matches do not have a corresponding Australian school-holiday city.
- Player-level statistics and forecast-weather data are outside the current v1 scope.

The next notebook will create the required PostgreSQL tables, load the five exported CSV files, and verify database row counts and key constraints. Feature engineering and model development will be completed in later stages.